# Checkpoint-only analyses for the revision (no training)

Loads the trained seed-42 SMGF-Net + MS-CA-ECA checkpoint (`A_full_smgfnet_mscaeca.pth`, written by the stage-1 notebook) or, **if no checkpoint exists, retrains it once** (§2, about the length of the stage-1 training run) and saves it. It then runs the analyses that need predictions but no retraining:

| Review point | Section | Output files |
|---|---|---|
| 62 external metrics on contaminated vs clean subsets, with bootstrap CIs (also answers the missing external CIs, point 63) | §4 | `ext_subset_metrics.csv`, `ext_predictions_*.npz` |
| 65 random-saliency baseline for deletion / insertion AUC | §5 | `xai_faithfulness_vs_random.csv` |
| 66 weight-randomization check: cascade, independent seeds, chance level, Score-CAM | §6 | `randomization_cascade.csv`, `randomization_summary.csv` |
| 68 attribution-vs-mask overlap (pointing game, top-15 % IoU, energy in mask) with random and centre baselines | §7 | `mask_overlap_*.csv` |
| AMEF check: which preprocessing reproduces the reported 95.28 % / 96.28 % | §3 | printed + `amef_check.csv` |

**Attach as Kaggle inputs:** BRISC2025 (as in stage-1), the four external datasets used in stage-1, and a dataset that contains
`A_full_smgfnet_mscaeca.pth` if you have one; with no checkpoint the notebook trains it. **Download the saved `.pth` at the end** (Output tab) so it is kept.


## §1 · Dependencies and definitions (reused unchanged from the stage-1 notebook)

In [ ]:
!pip install -q -U timm scikit-learn thop scikit-image


In [ ]:
import os, math, random, warnings, hashlib
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                              precision_score, recall_score, roc_auc_score,
                              confusion_matrix, classification_report,
                              matthews_corrcoef, cohen_kappa_score)
import matplotlib.pyplot as plt
import timm
import pandas as pd
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE    = 224
NUM_CLASSES = 4
CLASS_NAMES  = ["glioma", "meningioma", "notumor", "pituitary"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".jfif", ".tif", ".tiff", ".webp"}

OUT_DIR  = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE  = 32
NUM_WORKERS = 2

DROPOUT_RATE  = 0.3
USE_AMP       = True
EARLY_STOP_PATIENCE = 10   # epochs with no val AUC improvement before stopping

print("Device:", DEVICE)
print("Classes:", CLASS_NAMES)
print(f"DROPOUT_RATE={DROPOUT_RATE}  USE_AMP={USE_AMP}")


In [ ]:
USE_AMEF = False   # set automatically in §3 by checking which setting reproduces the reported accuracy
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def read_gray(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f"Cannot read: {path}")
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY if img.shape[-1] == 3
                           else cv2.COLOR_BGRA2GRAY)
    return img

def robust_normalize(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    p1, p99 = np.percentile(img, [1, 99])
    img = np.clip(img, p1, p99)
    img -= img.min(); img /= (img.max() + 1e-8)
    return img

def brain_mask(img01: np.ndarray) -> np.ndarray:
    """Otsu threshold + morphological cleanup + largest-connected-component
    to isolate brain tissue from black background/border."""
    img_u8 = (np.clip(img01, 0, 1) * 255).astype(np.uint8)
    _, mask = cv2.threshold(img_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num <= 1:
        return np.ones_like(img01, dtype=bool)
    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    brain = (labels == largest)
    if brain.sum() < 0.02 * img01.size:
        return np.ones_like(img01, dtype=bool)
    return brain


def laplacian_noise_estimate(gray_u8: np.ndarray) -> float:
    """Fast noise-sigma estimator (Immerkaer 1996) on an 8-bit grayscale image."""
    H, W = gray_u8.shape
    if H < 3 or W < 3:
        return 0.0
    kernel = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)
    conv = cv2.filter2D(gray_u8.astype(np.float32), -1, kernel)
    sigma = np.sqrt(np.pi / 2) * np.mean(np.abs(conv)) / (6 * (W - 2) * (H - 2) + 1e-8) * (W * H)
    return float(np.clip(sigma, 0.0, 50.0))


def adaptive_gamma(img01: np.ndarray, mask: np.ndarray, target: float = 0.5,
                    max_shift: float = 0.15, deadzone: float = 0.08) -> float:
    """REDESIGNED: gamma that nudges masked-brain mean intensity toward `target`,
    but only when it's meaningfully off (|mu - target| > deadzone) and only
    partway there (capped at max_shift). The original version forced every
    image's mean all the way to exactly 0.5 regardless of starting point,
    which erases inter-image/inter-class intensity variance that
    robust_normalize already preserved -- variance that can carry real
    diagnostic signal (tumor subtypes differ in characteristic T1 intensity).
    This version corrects only genuinely under/over-exposed images and leaves
    normally-exposed ones alone."""
    vals = img01[mask]
    if vals.size == 0:
        return 1.0
    mu = float(np.clip(vals.mean(), 1e-3, 1 - 1e-3))
    if abs(mu - target) <= deadzone:
        return 1.0  # already reasonable -- don't touch it
    bounded_target = mu + np.clip(target - mu, -max_shift, max_shift)
    gamma = np.log(bounded_target) / np.log(mu)
    return float(np.clip(gamma, 0.7, 1.4))  # tighter bound than original [0.5, 2.0]


def masked_entropy(img01: np.ndarray, mask: np.ndarray, bins: int = 32) -> float:
    vals = img01[mask]
    if vals.size == 0:
        return 0.0
    hist, _ = np.histogram(vals, bins=bins, range=(0, 1), density=True)
    hist = hist / (hist.sum() + 1e-8)
    hist = hist[hist > 0]
    return float(-np.sum(hist * np.log2(hist)))


def amef_preprocess(img01: np.ndarray, base_clip: float = 2.0, kappa: float = 0.05,
                     entropy_skip_threshold: float = 3.6) -> np.ndarray:
    """Adaptive MRI Enhancement Framework (v2): noise-gated CLAHE clip limit +
    bounded/deadzoned adaptive gamma correction + entropy-based CLAHE skip.
    Parameter-free at inference time (all thresholds fixed constants).
    Only the gamma step changed vs. v1 -- see adaptive_gamma docstring."""
    mask = brain_mask(img01)
    gray_u8 = (np.clip(img01, 0, 1) * 255).astype(np.uint8)

    sigma_noise = laplacian_noise_estimate(gray_u8)
    gamma = adaptive_gamma(img01, mask)
    img_gamma = np.clip(img01 ** gamma, 0, 1) if gamma != 1.0 else img01

    ent = masked_entropy(img_gamma, mask)
    if ent < entropy_skip_threshold:
        clip_limit = base_clip / (1.0 + kappa * sigma_noise)
        clahe = cv2.createCLAHE(clipLimit=max(clip_limit, 0.5), tileGridSize=(8, 8))
        img_u8 = (img_gamma * 255).astype(np.uint8)
        img_out = clahe.apply(img_u8).astype(np.float32) / 255.0
    else:
        img_out = img_gamma
    return img_out


def to_3ch_tensor(img01: np.ndarray) -> torch.Tensor:
    rgb = np.stack([img01, img01, img01], axis=2)
    rgb = torch.tensor(rgb).permute(2, 0, 1).float()
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (rgb - mean) / std


def preprocess(path: str) -> torch.Tensor:
    """Returns a single [3,224,224] image tensor.
    Pipeline: read_gray -> resize -> robust_normalize -> tensor.
    AMEF (adaptive gamma + entropy-gated CLAHE) is INTENTIONALLY NOT applied
    here. Two independent single-seed ablation runs (original AMEF, then a
    redesigned bounded/deadzoned adaptive_gamma) both showed AMEF losing to
    plain robust-normalize-only preprocessing on every metric (accuracy, F1,
    AUC, ECE) -- see section 21b. `amef_preprocess` and its helpers are kept
    below (unused by this function) only so that ablation cell stays
    reproducible; they are not part of the model's actual preprocessing path."""
    gray = read_gray(path)
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    img01 = robust_normalize(gray)
    if USE_AMEF:
        img01 = amef_preprocess(img01)
    return to_3ch_tensor(img01)


# --- Diagnostic utility (call manually, not part of the pipeline) ---
def amef_diagnostic(sample_paths, n=200):
    """Reports how often gamma correction actually fires and by how much,
    plus CLAHE trigger rate. Run before/after any AMEF change to confirm
    the fix is doing what's intended, and to sanity-check against the old
    version's behavior."""
    gammas, mu_before, mu_after, clahe_fired = [], [], [], []
    for p in sample_paths[:n]:
        gray = cv2.resize(read_gray(p), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        img01 = robust_normalize(gray)
        mask = brain_mask(img01)
        mu_before.append(float(img01[mask].mean()))
        gamma = adaptive_gamma(img01, mask)
        gammas.append(gamma)
        img_gamma = np.clip(img01 ** gamma, 0, 1) if gamma != 1.0 else img01
        mu_after.append(float(img_gamma[mask].mean()))
        clahe_fired.append(masked_entropy(img_gamma, mask) < 3.6)

    gammas = np.array(gammas)
    print(f"gamma == 1.0 (untouched): {100*np.mean(gammas == 1.0):.1f}% of images")
    print(f"gamma (when applied):     mean={gammas[gammas!=1.0].mean() if (gammas!=1.0).any() else float('nan'):.3f}")
    print(f"mean-intensity before gamma: {np.mean(mu_before):.3f} ± {np.std(mu_before):.3f}")
    print(f"mean-intensity after  gamma: {np.mean(mu_after):.3f} ± {np.std(mu_after):.3f}")
    print(f"CLAHE fired on {100*np.mean(clahe_fired):.1f}% of images")

print("Preprocessing + AMEF (v2 - bounded/deadzoned gamma) ready.")

In [ ]:
import torchvision.transforms.functional as TF

class MRIAugment:
    def __init__(self, p_flip=0.5, max_rotate=10.0, p_rotate=0.5,
                 brightness=0.2, contrast=0.2, p_jitter=0.5,
                 translate=0.06, p_translate=0.3,
                 use_gamma=True, gamma_range=(0.85, 1.15), p_gamma=0.3):
        self.p_flip = p_flip
        self.max_rotate = max_rotate; self.p_rotate = p_rotate
        self.brightness = brightness; self.contrast = contrast; self.p_jitter = p_jitter
        self.translate = translate; self.p_translate = p_translate
        self.use_gamma = use_gamma; self.gamma_range = gamma_range; self.p_gamma = p_gamma

    def __call__(self, img: torch.Tensor) -> torch.Tensor:
        if random.random() < self.p_flip:
            img = TF.hflip(img)
        if random.random() < self.p_rotate:
            angle = random.uniform(-self.max_rotate, self.max_rotate)
            img = TF.rotate(img, angle)
        if random.random() < self.p_translate:
            tx = random.uniform(-self.translate, self.translate) * IMG_SIZE
            ty = random.uniform(-self.translate, self.translate) * IMG_SIZE
            img = TF.affine(img, angle=0, translate=[int(tx), int(ty)], scale=1.0, shear=0)
        if random.random() < self.p_jitter:
            b = 1.0 + random.uniform(-self.brightness, self.brightness)
            c = 1.0 + random.uniform(-self.contrast, self.contrast)
            img = torch.clamp(img * c + (b - 1.0), -3.0, 3.0)
        if self.use_gamma and random.random() < self.p_gamma:
            gamma = random.uniform(*self.gamma_range)
            img = torch.sign(img) * (torch.abs(img) ** gamma)
        return img

train_augment = MRIAugment()
print("MRIAugment ready.")


In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, samples: List[Dict], is_train: bool = False,
                 augmenter: Optional[MRIAugment] = None):
        self.samples   = samples
        self.is_train  = is_train
        self.augmenter = augmenter

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = preprocess(s["path"])
        if self.is_train and self.augmenter is not None:
            img = self.augmenter(img)
        return {"image": img, "label": torch.tensor(s["label"], dtype=torch.long),
                "class_name": s["class_name"], "path": s["path"]}

print("BrainTumorDataset ready.")


In [ ]:
class ECA(nn.Module):
    """Efficient Channel Attention (Wang et al., 2020). ~5-9 parameters."""
    def __init__(self, channels: int, gamma: int = 2, b: int = 1):
        super().__init__()
        t = int(abs((math.log2(channels) + b) / gamma))
        k = t if t % 2 else t + 1
        k = max(k, 3)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=(k - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y)
        y = self.sigmoid(y).transpose(-1, -2).unsqueeze(-1)
        return x * y.expand_as(x)


class DWSepBlock(nn.Module):
    """Depthwise-separable conv block: DW3x3 -> BN -> act -> PW1x1 -> BN -> act,
    with an optional stride-2 downsample and a residual connection when shapes match."""
    def __init__(self, c_in: int, c_out: int, stride: int = 1, act=nn.SiLU):
        super().__init__()
        self.dw = nn.Conv2d(c_in, c_in, 3, stride=stride, padding=1, groups=c_in, bias=False)
        self.bn1 = nn.BatchNorm2d(c_in)
        self.pw = nn.Conv2d(c_in, c_out, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.act = act(inplace=True)
        self.use_res = (stride == 1 and c_in == c_out)

    def forward(self, x):
        identity = x
        x = self.act(self.bn1(self.dw(x)))
        x = self.bn2(self.pw(x))
        if self.use_res:
            x = x + identity
        return self.act(x)


class SpatialBackbone(nn.Module):
    """Lightweight from-scratch depthwise-separable stem + 3 stages. Exposes
    F1/F2/F3 (stage outputs, strides /4, /8, /16) for multi-scale fusion --
    deliberately NOT a pretrained off-the-shelf backbone."""
    def __init__(self, stem_ch=32, c1=48, c2=88, c3=160, depth=(1, 2, 2)):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, stem_ch, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(stem_ch),
            nn.SiLU(inplace=True),
        )
        self.stage1 = self._make_stage(stem_ch, c1, depth[0])
        self.stage2 = self._make_stage(c1, c2, depth[1])
        self.stage3 = self._make_stage(c2, c3, depth[2])
        self.out_channels = [c1, c2, c3]

    @staticmethod
    def _make_stage(c_in, c_out, n_extra):
        layers = [DWSepBlock(c_in, c_out, stride=2)]
        for _ in range(n_extra):
            layers.append(DWSepBlock(c_out, c_out, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)          # /2
        f1 = self.stage1(x)       # /4
        f2 = self.stage2(f1)      # /8
        f3 = self.stage3(f2)      # /16
        return f1, f2, f3


class MultiScaleSpatial(nn.Module):
    """Spatial representation Fs: projects F1/F2/F3 to a common width, aligns
    to F3's resolution, fuses with learned global softmax weights, then a
    depthwise residual refine."""
    def __init__(self, in_channels_list: List[int], out_channels: int):
        super().__init__()
        self.proj = nn.ModuleList([
            nn.Conv2d(c, out_channels, 1, bias=False) for c in in_channels_list
        ])
        self.bn = nn.ModuleList([nn.BatchNorm2d(out_channels) for _ in in_channels_list])
        self.scale_weight = nn.Parameter(torch.ones(len(in_channels_list)))
        self.refine = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1, groups=out_channels, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, feats: List[torch.Tensor]) -> torch.Tensor:
        target_hw = feats[-1].shape[-2:]
        aligned = [F.adaptive_avg_pool2d(bn(p(f)), target_hw)
                   for p, bn, f in zip(self.proj, self.bn, feats)]
        alpha = torch.softmax(self.scale_weight, dim=0)
        fused = sum(a * x for a, x in zip(alpha, aligned))
        return fused + self.refine(fused)


class MorphologyBranch(nn.Module):
    """Morphology-aware representation Fm. A fixed, parameter-free
    morphological-gradient operator (dilation - erosion via max-pool) applied
    to F3, then a learnable depthwise+pointwise transform, trained end-to-end."""
    def __init__(self, c_in: int, out_channels: int):
        super().__init__()
        self.pool = nn.MaxPool2d(3, stride=1, padding=1)
        self.dw = nn.Conv2d(c_in, c_in, 3, padding=1, groups=c_in, bias=False)
        self.bn1 = nn.BatchNorm2d(c_in)
        self.pw = nn.Conv2d(c_in, out_channels, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        dilation = self.pool(x)
        erosion = -self.pool(-x)
        grad = dilation - erosion          # morphological gradient, parameter-free
        x = self.act(self.bn1(self.dw(grad)))
        x = self.act(self.bn2(self.pw(x)))
        return x


class GatedFusion2(nn.Module):
    """Two-branch per-channel gated fusion (Spatial + Morphology). A
    per-channel, per-sample gate learned from both branches' pooled
    statistics -- gives the network the capacity to weight branches
    differently per output channel and per input image."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        hidden = max(channels // reduction, 8)
        self.gate = nn.Sequential(
            nn.Linear(channels * 2, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels * 2),
        )

    def forward(self, fs: torch.Tensor, fm: torch.Tensor):
        b, c = fs.shape[0], fs.shape[1]
        pooled = torch.cat([self.pool(fs).flatten(1), self.pool(fm).flatten(1)], dim=1)
        raw = self.gate(pooled).view(b, 2, c)
        weights = torch.softmax(raw, dim=1)          # per-channel, per-sample, sums to 1 over the 2 branches
        fused = (weights[:, 0, :].view(b, c, 1, 1) * fs +
                 weights[:, 1, :].view(b, c, 1, 1) * fm)
        return fused, weights


class SimpleAvgFusion(nn.Module):
    """Parameter-free 50/50 averaging fusion. Used ONLY in ablation config D
    ('remove Gated Fusion') so the model still has some way to combine
    Fs/Fm -- not part of the proposed architecture itself."""
    def forward(self, fs: torch.Tensor, fm: torch.Tensor):
        fused = 0.5 * fs + 0.5 * fm
        b, c = fs.shape[0], fs.shape[1]
        weights = torch.full((b, 2, c), 0.5, device=fs.device)
        return fused, weights


class SmallHead(nn.Module):
    """GAP -> small projection -> BatchNorm1d -> GELU -> Dropout -> Linear(4)."""
    def __init__(self, in_dim: int, hidden_dim: int, num_classes: int, dropout: float):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.gap(x).flatten(1)
        emb = self.act(self.bn(self.fc1(x)))
        out = self.fc2(self.drop(emb))
        return out, emb

print("Shared building blocks ready (Spatial branch, Morphology branch, Gated Fusion — unchanged).")


In [ ]:
class MultiScaleContext(nn.Module):
    """Cheap multi-scale contextual extraction: channel-reduce with 1x1, three
    parallel DEPTHWISE 3x3 convs at dilation 1/2/3 (receptive fields 3/5/7),
    concat, expand back with 1x1. No full (non-depthwise, non-1x1) convs, per
    the lightweight-operations requirement."""
    def __init__(self, channels: int, reduction: int = 18):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.reduce = nn.Conv2d(channels, hidden, 1, bias=False)
        self.bn_reduce = nn.BatchNorm2d(hidden)
        self.dw1 = nn.Conv2d(hidden, hidden, 3, padding=1, dilation=1, groups=hidden, bias=False)
        self.dw2 = nn.Conv2d(hidden, hidden, 3, padding=2, dilation=2, groups=hidden, bias=False)
        self.dw3 = nn.Conv2d(hidden, hidden, 3, padding=3, dilation=3, groups=hidden, bias=False)
        self.bn_cat = nn.BatchNorm2d(hidden * 3)
        self.expand = nn.Conv2d(hidden * 3, channels, 1, bias=False)
        self.bn_expand = nn.BatchNorm2d(channels)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        r = self.act(self.bn_reduce(self.reduce(x)))
        m = torch.cat([self.dw1(r), self.dw2(r), self.dw3(r)], dim=1)
        m = self.act(self.bn_cat(m))
        return self.bn_expand(self.expand(m))


class CoordinateAttention(nn.Module):
    """Coordinate Attention (Hou et al., 2021), lightweight form: separately
    pools along H and W to preserve direction-specific information, encodes
    both with a shared 1x1 reduction, then splits back into an H-gate and a
    W-gate (both 1x1 projections, no full convolutions)."""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.reduce = nn.Conv2d(channels, hidden, 1, bias=False)
        self.bn = nn.BatchNorm2d(hidden)
        self.act = nn.SiLU(inplace=True)
        self.conv_h = nn.Conv2d(hidden, channels, 1, bias=False)
        self.conv_w = nn.Conv2d(hidden, channels, 1, bias=False)

    def forward(self, x):
        b, c, h, w = x.shape
        x_h = self.pool_h(x)                          # (b,c,h,1) -- height-direction info
        x_w = self.pool_w(x).permute(0, 1, 3, 2)       # (b,c,w,1) -- width-direction info
        y = torch.cat([x_h, x_w], dim=2)               # (b,c,h+w,1)
        y = self.act(self.bn(self.reduce(y)))
        y_h, y_w = torch.split(y, [h, w], dim=2)
        y_w = y_w.permute(0, 1, 3, 2)                  # back to (b,hidden,1,w)
        g_h = torch.sigmoid(self.conv_h(y_h))          # (b,c,h,1)
        g_w = torch.sigmoid(self.conv_w(y_w))          # (b,c,1,w)
        return x * g_h * g_w


class MSCAECA(nn.Module):
    """Multi-Scale Coordinate-ECA Attention (proposed). Fused feature ->
    parallel {MultiScaleContext, CoordinateAttention} -> ECA channel
    refinement -> residual. use_multiscale / use_coord / use_eca gate each
    stage independently for the internal-ablation study (F-I, section 16);
    a disabled stage is an identity pass-through, so this ONE class serves
    both the full module and every internal-ablation variant -- there is no
    separate ECA/SE/CBAM comparison code."""
    def __init__(self, channels: int, ms_reduction: int = 18, coord_reduction: int = 16,
                 use_multiscale: bool = True, use_coord: bool = True, use_eca: bool = True):
        super().__init__()
        self.use_multiscale = use_multiscale
        self.use_coord = use_coord
        self.use_eca = use_eca
        self.ms = MultiScaleContext(channels, reduction=ms_reduction) if use_multiscale else None
        self.coord = CoordinateAttention(channels, reduction=coord_reduction) if use_coord else None
        self.eca = ECA(channels) if use_eca else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = x
        if self.ms is not None:
            out = out + self.ms(out)
        if self.coord is not None:
            out = self.coord(out)
        if self.eca is not None:
            out = self.eca(out)
        return out + identity

print("MS-CA-ECA (Multi-Scale Coordinate-ECA Attention) ready.")


In [ ]:
# Channel configuration -- rep_width narrowed from SMGF-Net's 160 to 148 to make
# room for MS-CA-ECA under the same 211,560-parameter ceiling (see markdown above).
# Spatial branch / Morphology branch / Gated Fusion configs are otherwise IDENTICAL
# to the reference SMGF-Net.
SMGF_MSCAECA_CONFIG = dict(stem_ch=32, c1=48, c2=88, c3=160, depth=(1, 2, 2),
                            rep_width=148, head_hidden=128)
MSCAECA_CONFIG = dict(ms_reduction=18, coord_reduction=16)
REFERENCE_SMGFNET_PARAMS = 211_560   # previous model (plain ECA), for the required parameter-diff report
PARAM_BUDGET = 211_560               # hard constraint: MS-CA-ECA SMGF-Net <= this


class SMGFNetMSCAECA(nn.Module):
    """SMGF-Net + MS-CA-ECA -- the final proposed architecture, and the ONLY
    model class used throughout this notebook (full model + every ablation
    variant, selected via the flags below).

    Input MRI -> SpatialBackbone (stem + 3 depthwise-separable stages,
    exposing F1/F2/F3) -> {MultiScaleSpatial(Fs), MorphologyBranch(Fm)} ->
    GatedFusion2 -> MS-CA-ECA -> SmallHead -> 4-class logits.

    use_spatial / use_morphology / use_gated_fusion / attention: component
    ablation (section 15, configs A-E). ms_kwargs (use_multiscale /
    use_coord / use_eca): MS-CA-ECA internal ablation (section 16, configs
    F-I). Spatial branch, Morphology branch and Gated Fusion are NOT
    redesigned for any of these ablations -- only removed/replaced wholesale,
    exactly as the task brief specifies.
    """
    def __init__(self, num_classes: int = NUM_CLASSES, dropout: float = DROPOUT_RATE,
                 stem_ch: int = 32, c1: int = 48, c2: int = 88, c3: int = 160,
                 depth=(1, 2, 2), rep_width: int = 148, head_hidden: int = 128,
                 use_spatial: bool = True, use_morphology: bool = True,
                 use_gated_fusion: bool = True, attention: str = "msca_eca",
                 ms_kwargs: Optional[Dict[str, Any]] = None):
        super().__init__()
        assert use_spatial or use_morphology, "at least one of Spatial/Morphology must be active"
        self.use_spatial = use_spatial
        self.use_morphology = use_morphology
        self.use_gated_fusion = use_gated_fusion
        self.attention_type = attention
        ms_kwargs = {**MSCAECA_CONFIG, **(ms_kwargs or {})}

        self.backbone = SpatialBackbone(stem_ch, c1, c2, c3, depth=depth)
        self.multiscale = MultiScaleSpatial(self.backbone.out_channels, rep_width) if use_spatial else None
        self.morphology = MorphologyBranch(c3, rep_width) if use_morphology else None

        if use_spatial and use_morphology:
            self.fusion = GatedFusion2(channels=rep_width) if use_gated_fusion else SimpleAvgFusion()
        else:
            self.fusion = None   # single-branch ablation (B/C): nothing to fuse

        if attention == "msca_eca":
            self.attn = MSCAECA(rep_width, **ms_kwargs)
        elif attention == "eca":
            self.attn = ECA(rep_width)
        elif attention == "none":
            self.attn = nn.Identity()
        else:
            raise ValueError(f"unknown attention type: {attention}")

        self.head = SmallHead(rep_width, head_hidden, num_classes, dropout)

    def forward(self, x: torch.Tensor, return_all: bool = False) -> Dict[str, torch.Tensor]:
        f1, f2, f3 = self.backbone(x)
        fs = self.multiscale([f1, f2, f3]) if self.use_spatial else None
        fm = self.morphology(f3) if self.use_morphology else None

        weights = None
        if self.use_spatial and self.use_morphology:
            fused, weights = self.fusion(fs, fm)
        elif self.use_spatial:
            fused = fs
        else:
            fused = fm

        attn = self.attn(fused)
        logits, emb = self.head(attn)
        out = {"logits": logits, "emb": emb}
        if return_all:
            out.update({"f1": f1, "f2": f2, "f3": f3, "spatial": fs, "morphology": fm,
                        "fusion_weights": weights, "fused": fused, "attn": attn})
        return out


def build_smgf_mscaeca(**kwargs) -> SMGFNetMSCAECA:
    """Builds SMGF-Net + MS-CA-ECA (or an ablation variant of it). Any of
    SMGFNetMSCAECA's constructor args can be overridden via kwargs, e.g.
    build_smgf_mscaeca(use_spatial=False) or
    build_smgf_mscaeca(ms_kwargs=dict(use_coord=False))."""
    cfg = {**SMGF_MSCAECA_CONFIG}
    ms_override = kwargs.pop("ms_kwargs", None)
    cfg.update(kwargs)
    if ms_override is not None:
        cfg["ms_kwargs"] = ms_override
    model = SMGFNetMSCAECA(**cfg).to(DEVICE)
    return model


def param_breakdown(model: "SMGFNetMSCAECA") -> Dict[str, int]:
    def count(m):
        return 0 if m is None else sum(p.numel() for p in m.parameters() if p.requires_grad)
    parts = {
        "backbone (stem + 3 DW-sep stages)": count(model.backbone),
        "spatial branch (F1/F2/F3 -> Fs)": count(model.multiscale),
        "morphology branch (Fm)": count(model.morphology),
        "gated fusion (Fs, Fm -> fused)": count(model.fusion),
        "MS-CA-ECA attention": count(model.attn),
        "classifier head": count(model.head),
    }
    parts["TOTAL"] = sum(v for k, v in parts.items())
    return parts


print("build_smgf_mscaeca() ready.")


In [ ]:
def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15):
    """Returns (ece, bin_accs, bin_confs, bin_counts)."""
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(np.float32)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_accs, bin_confs, bin_counts = [], [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        count = mask.sum()
        if count == 0:
            bin_accs.append(0); bin_confs.append(0); bin_counts.append(0)
            continue
        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += (count / len(labels)) * abs(bin_acc - bin_conf)
        bin_accs.append(bin_acc); bin_confs.append(bin_conf); bin_counts.append(count)
    return ece, bin_accs, bin_confs, bin_counts


def compute_class_weights(samples: List[Dict], num_classes: int = NUM_CLASSES) -> torch.Tensor:
    """Inverse-frequency class weights, normalized to mean 1.0."""
    counts = np.zeros(num_classes)
    for s in samples:
        counts[s["label"]] += 1
    counts = np.maximum(counts, 1)
    weights = counts.sum() / (num_classes * counts)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


class CAWLPlus(nn.Module):
    def __init__(self, class_weights: torch.Tensor, num_classes: int = NUM_CLASSES,
                 emb_dim: int = 64, label_smoothing: float = 0.05,
                 target_confidence_cap: float = 0.85, lambda_max: float = 0.3,
                 anneal_epochs: int = 15, proto_momentum: float = 0.9,
                 proto_margin: float = 0.5, beta: float = 0.1,
                 proto_warmup_epochs: int = 8):
        super().__init__()
        self.register_buffer("class_weights", class_weights)
        self.label_smoothing = label_smoothing
        self.target_confidence_cap = target_confidence_cap
        self.lambda_max = lambda_max
        self.anneal_epochs = anneal_epochs
        self.current_epoch = 0

        self.num_classes = num_classes
        self.proto_momentum = proto_momentum
        self.proto_margin = proto_margin
        self.beta = beta
        self.proto_warmup_epochs = proto_warmup_epochs
        self.register_buffer("prototypes", torch.zeros(num_classes, emb_dim))
        self.register_buffer("proto_initialized", torch.zeros(num_classes, dtype=torch.bool))

    def _lambda_t(self):
        frac = min(self.current_epoch / max(self.anneal_epochs, 1), 1.0)
        return self.lambda_max * frac

    @torch.no_grad()
    def _update_prototypes(self, emb: torch.Tensor, targets: torch.Tensor):
        for c in range(self.num_classes):
            mask = targets == c
            if mask.sum() == 0:
                continue
            batch_mean = emb[mask].mean(dim=0)
            if not self.proto_initialized[c]:
                self.prototypes[c] = batch_mean
                self.proto_initialized[c] = True
            else:
                self.prototypes[c] = (self.proto_momentum * self.prototypes[c] +
                                       (1 - self.proto_momentum) * batch_mean)

    def _prototype_margin_loss(self, emb: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if not self.proto_initialized.all():
            return torch.tensor(0.0, device=emb.device)
        protos = self.prototypes.to(emb.device)
        d = torch.cdist(emb, protos) ** 2
        d_pos = d.gather(1, targets.view(-1, 1)).squeeze(1)
        d_masked = d.clone()
        d_masked.scatter_(1, targets.view(-1, 1), float("inf"))
        d_neg = d_masked.min(dim=1).values
        return F.relu(d_pos - d_neg + self.proto_margin).mean()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor,
                emb: Optional[torch.Tensor] = None) -> torch.Tensor:
        ce = F.cross_entropy(logits, targets, weight=self.class_weights,
                              label_smoothing=self.label_smoothing)
        probs = F.softmax(logits, dim=1)
        conf, _ = probs.max(dim=1)
        pred = logits.argmax(dim=1)
        correct = (pred == targets).float()
        overconf = F.relu(conf - self.target_confidence_cap) * correct
        conf_penalty = overconf.mean()
        loss = ce + self._lambda_t() * conf_penalty

        if emb is not None and self.training:
            self._update_prototypes(emb.detach(), targets)   # EMA warms up silently from epoch 1
        if emb is not None and self.current_epoch >= self.proto_warmup_epochs:
            loss = loss + self.beta * self._prototype_margin_loss(emb, targets)  # gated
        return loss

print("CAWLPlus ready (with proto_warmup_epochs gate).")


In [ ]:
def _file_hash(path: str) -> str:
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def dedupe_splits(splits: Dict[str, List[Dict]],
                   priority: List[str] = ["test", "val", "train"]) -> Dict[str, List[Dict]]:
    hash_to_owner: Dict[str, Tuple[str, int]] = {}
    to_remove: Dict[str, set] = {name: set() for name in splits}
    n_cross = n_intra = 0

    for name in sorted(splits, key=lambda n: priority.index(n) if n in priority else 99):
        for idx, s in enumerate(splits[name]):
            h = _file_hash(s["path"])
            if h in hash_to_owner:
                owner_name, owner_idx = hash_to_owner[h]
                if owner_name == name:
                    n_intra += 1
                    if n_intra <= 5:
                        print(f"  [DUPLICATE within '{name}'] {splits[name][owner_idx]['path']}  ==  {s['path']}")
                else:
                    n_cross += 1
                    if n_cross <= 5:
                        print(f"  [LEAKAGE: kept in '{owner_name}', removed from '{name}'] {s['path']}")
                to_remove[name].add(idx)
            else:
                hash_to_owner[h] = (name, idx)

    cleaned = {name: [s for i, s in enumerate(samples) if i not in to_remove[name]]
               for name, samples in splits.items()}
    print(f"[LEAKAGE CHECK] {n_cross} cross-split leak(s) + {n_intra} intra-split "
          f"duplicate(s) auto-removed ({n_cross + n_intra} total).")
    for name in splits:
        removed = len(to_remove[name])
        if removed:
            print(f"  {name}: {len(splits[name])} -> {len(cleaned[name])} ({removed} removed)")
    return cleaned


def scan_dataset_generic(root: Path, folder_to_class: Dict[str, str], split_name: str) -> List[Dict[str, Any]]:
    samples = []
    for folder_name, cls in folder_to_class.items():
        cls_dir = root / folder_name
        if not cls_dir.exists():
            print(f"  [WARN] missing: {cls_dir}")
            continue
        n_before = len(samples)
        for p in cls_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                samples.append({"path": str(p), "label": CLASS_TO_IDX[cls],
                                 "class_name": cls, "split": split_name})
        print(f"  {split_name}/{folder_name} -> {cls}: {len(samples) - n_before} images")
    print(f"{split_name}: {len(samples)} images across {NUM_CLASSES} classes")
    return samples

print("Scanner + leakage-check utilities ready.")


In [ ]:
BRISC_ROOT = Path("/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task")
BRISC_TRAIN_ROOT = BRISC_ROOT / "train"
BRISC_TEST_ROOT  = BRISC_ROOT / "test"
print(f"BRISC2025 train exists: {BRISC_TRAIN_ROOT.exists()} -> {BRISC_TRAIN_ROOT}")
print(f"BRISC2025 test  exists: {BRISC_TEST_ROOT.exists()} -> {BRISC_TEST_ROOT}")
if not (BRISC_TRAIN_ROOT.exists() and BRISC_TEST_ROOT.exists()):
    raise RuntimeError("BRISC2025 not found. Make sure it's attached to this "
                        "Kaggle notebook (Add Input -> search 'brisc2025').")

BRISC_FOLDER_TO_CLASS = {
    "glioma": "glioma", "meningioma": "meningioma",
    "no_tumor": "notumor", "pituitary": "pituitary",
}


In [ ]:
TOTAL_EPOCHS  = 50
WEIGHT_DECAY  = 2e-4
GRAD_CLIP     = 1.0
PEAK_LR       = 1.8e-4   # was 4e-4 -- too high a peak was a major source of curve noise
MIN_LR        = 1e-6

WARMUP_EPOCHS_V2     = 15   # was 10 -- longer, gentler ramp instead of a hard 10-epoch climb
WARMUP_START_DIV_V2  = 50

FREEZE_BACKBONE_EPOCHS = 3

# Fine-grained early logging: for the first FINEGRAIN_EPOCHS epochs, evaluate
# train(clean)/val this many times DURING the epoch (not just once at the end).
# A frozen-backbone linear-probe head converges fast -- within a single epoch it can
# already climb from chance-level (~25% for 4 classes) most of the way to its plateau.
# Logging only once per epoch was hiding that climb and making it look like val
# "jumped" straight to a high number with no visible ramp. This makes the ramp visible.
FINEGRAIN_EPOCHS   = 2
FINEGRAIN_SUBSTEPS = 5

def lr_at_epoch_v2(epoch: int) -> float:
    warmup_start_lr = PEAK_LR / WARMUP_START_DIV_V2
    if epoch <= WARMUP_EPOCHS_V2:
        frac = epoch / WARMUP_EPOCHS_V2
        return warmup_start_lr + frac * (PEAK_LR - warmup_start_lr)
    remaining_epochs = TOTAL_EPOCHS - WARMUP_EPOCHS_V2
    progress = min((epoch - WARMUP_EPOCHS_V2) / max(remaining_epochs, 1), 1.0)
    return MIN_LR + 0.5 * (PEAK_LR - MIN_LR) * (1 + math.cos(math.pi * progress))


def train_one_epoch_v2(model, loader, optimizer, criterion: CAWLPlus,
                        scaler: Optional[GradScaler] = None):
    model.train()
    total_loss = correct = total = 0
    use_amp = scaler is not None
    for batch in loader:
        img   = batch["image"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with autocast():
                out  = model(img)
                loss = criterion(out["logits"], label, out["emb"])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(img)
            loss = criterion(out["logits"], label, out["emb"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        pred = out["logits"].argmax(1)
        correct += (pred == label).sum().item()
        total   += label.size(0)
        total_loss += loss.item() * label.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_v2(model, loader, criterion: CAWLPlus, device: str = DEVICE) -> Dict[str, Any]:
    model.eval()
    all_preds, all_labels, all_probs, total_loss = [], [], [], 0.0
    for batch in loader:
        img   = batch["image"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)
        out   = model(img)
        loss  = criterion(out["logits"], label, out["emb"])
        total_loss += loss.item() * label.size(0)
        probs = torch.softmax(out["logits"], dim=1).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_preds.extend(out["logits"].argmax(1).cpu().tolist())
        all_labels.extend(label.cpu().tolist())

    n = len(all_labels)
    y_true = np.array(all_labels); y_pred = np.array(all_preds); y_prob = np.array(all_probs)
    try:
        auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auc = float("nan")
    cm = confusion_matrix(y_true, y_pred)
    ece, _, _, _ = expected_calibration_error(y_prob, y_true)

    return {
        "loss": total_loss / n,
        "acc": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "precision_mac": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_mac": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_mac": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "auc": auc,
        "mcc": matthews_corrcoef(y_true, y_pred),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "ece": ece,
        "cm": cm,
        "report": classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0),
    }


def plot_confusion_matrix(cm: np.ndarray, title: str = "Confusion Matrix", save_path: Optional[str] = None):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.colorbar(im, ax=ax); plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

# Fine-grained ("chunked") epoch trainer -- identical math to train_one_epoch_v2,
# but pauses n_chunks-1 times during the epoch to log a clean train/val snapshot at
# that fractional-epoch point. Only used for the first FINEGRAIN_EPOCHS epochs.
def train_one_epoch_chunked_v2(model, loader, optimizer, criterion: CAWLPlus,
                                scaler: Optional[GradScaler], n_chunks: int, epoch: int,
                                train_eval_loader, val_loader, history: Dict[str, list]):
    model.train()
    use_amp = scaler is not None
    batches = list(loader)
    n_batches = len(batches)
    chunk_size = max(1, n_batches // n_chunks)
    total_loss = correct = total = 0

    for i, batch in enumerate(batches):
        img   = batch["image"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with autocast():
                out  = model(img)
                loss = criterion(out["logits"], label, out["emb"])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(img)
            loss = criterion(out["logits"], label, out["emb"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        pred = out["logits"].argmax(1)
        correct += (pred == label).sum().item()
        total   += label.size(0)
        total_loss += loss.item() * label.size(0)

        is_chunk_boundary = ((i + 1) % chunk_size == 0) and (i + 1) < n_batches
        if is_chunk_boundary:
            frac_epoch = (epoch - 1) + (i + 1) / n_batches
            train_clean = evaluate_v2(model, train_eval_loader, criterion)
            val_metrics = evaluate_v2(model, val_loader, criterion)
            print(f"  epoch {frac_epoch:06.2f} (substep)  "
                  f"train_acc(clean)={train_clean['acc']:.4f}  "
                  f"val_acc={val_metrics['acc']:.4f}  val_auc={val_metrics['auc']:.4f}")
            history["epoch"].append(frac_epoch)
            history["train_loss"].append(train_clean["loss"])
            history["train_acc"].append(train_clean["acc"])
            history["train_loss_aug"].append(total_loss / total)
            history["train_acc_aug"].append(correct / total)
            history["val_loss"].append(val_metrics["loss"])
            history["val_acc"].append(val_metrics["acc"])
            history["val_auc"].append(val_metrics["auc"])
            history["val_ece"].append(val_metrics["ece"])
            history["lr"].append(optimizer.param_groups[0]["lr"])
            model.train()

    return total_loss / total, correct / total


print(f"Schedule ready: {TOTAL_EPOCHS} epochs, {WARMUP_EPOCHS_V2}-epoch gentle warmup "
      f"(start={PEAK_LR/WARMUP_START_DIV_V2:.1e}), peak_lr={PEAK_LR:.1e}, AMP={USE_AMP}, "
      f"early_stop_patience={EARLY_STOP_PATIENCE}, freeze_backbone_epochs={FREEZE_BACKBONE_EPOCHS}, "
      f"finegrain_epochs={FINEGRAIN_EPOCHS}x{FINEGRAIN_SUBSTEPS}")


In [ ]:
class ModelEMA:
    """Exponential moving average of the model's weights."""
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


def mixup_batch(img: torch.Tensor, label: torch.Tensor, alpha: float = 0.2):
    """Standard Mixup (Zhang et al., 2018)."""
    if alpha <= 0:
        return img, label, label, 1.0
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(img.size(0), device=img.device)
    mixed = lam * img + (1 - lam) * img[idx]
    return mixed, label, label[idx], lam


def mixup_cawl_loss(criterion: CAWLPlus, logits: torch.Tensor, emb: torch.Tensor,
                     y_a: torch.Tensor, y_b: torch.Tensor, lam: float) -> torch.Tensor:
    """Mixup-safe re-implementation of CAWLPlus.forward() (§9): computes CE +
    confidence-penalty for both mixed labels, but touches the prototype
    buffer only once per batch (dominant label), to avoid the in-place
    prototype-mutation issue of calling criterion(...) twice."""
    ce_a = F.cross_entropy(logits, y_a, weight=criterion.class_weights,
                            label_smoothing=criterion.label_smoothing)
    ce_b = F.cross_entropy(logits, y_b, weight=criterion.class_weights,
                            label_smoothing=criterion.label_smoothing)
    ce = lam * ce_a + (1 - lam) * ce_b

    probs = F.softmax(logits, dim=1)
    conf, _ = probs.max(dim=1)
    pred = logits.argmax(dim=1)
    correct = lam * (pred == y_a).float() + (1 - lam) * (pred == y_b).float()
    conf_penalty = (F.relu(conf - criterion.target_confidence_cap) * correct).mean()
    loss = ce + criterion._lambda_t() * conf_penalty

    y_dom = y_a if lam >= 0.5 else y_b
    if criterion.training:
        criterion._update_prototypes(emb.detach(), y_dom)
    if criterion.current_epoch >= criterion.proto_warmup_epochs:
        loss = loss + criterion.beta * criterion._prototype_margin_loss(emb, y_dom)
    return loss


def train_one_epoch_mixup(model, loader, optimizer, criterion: CAWLPlus,
                           scaler: Optional[GradScaler] = None,
                           ema: Optional[ModelEMA] = None,
                           mixup_alpha: float = 0.2):
    model.train()
    total_loss = correct = total = 0
    use_amp = scaler is not None
    for batch in loader:
        img   = batch["image"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        img_m, y_a, y_b, lam = mixup_batch(img, label, alpha=mixup_alpha)
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with autocast():
                out  = model(img_m)
                loss = mixup_cawl_loss(criterion, out["logits"], out["emb"], y_a, y_b, lam)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(img_m)
            loss = mixup_cawl_loss(criterion, out["logits"], out["emb"], y_a, y_b, lam)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        if ema is not None:
            ema.update(model)

        pred = out["logits"].argmax(1)
        correct += (lam * (pred == y_a).float() + (1 - lam) * (pred == y_b).float()).sum().item()
        total   += label.size(0)
        total_loss += loss.item() * label.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_v2_tta(model, loader, criterion: CAWLPlus, device: str = DEVICE) -> Dict[str, Any]:
    """Same metric suite as evaluate_v2 (§12), but averages softmax
    probabilities over the original image and its horizontal flip
    (test-time augmentation) before taking the argmax."""
    model.eval()
    all_preds, all_labels, all_probs, total_loss = [], [], [], 0.0
    for batch in loader:
        img   = batch["image"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)
        out1  = model(img)
        out2  = model(torch.flip(img, dims=[3]))
        probs = (torch.softmax(out1["logits"], dim=1) + torch.softmax(out2["logits"], dim=1)) / 2
        loss  = criterion(out1["logits"], label, out1["emb"])
        total_loss += loss.item() * label.size(0)
        all_probs.extend(probs.cpu().numpy().tolist())
        all_preds.extend(probs.argmax(1).cpu().tolist())
        all_labels.extend(label.cpu().tolist())

    n = len(all_labels)
    y_true = np.array(all_labels); y_pred = np.array(all_preds); y_prob = np.array(all_probs)
    try:
        auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auc = float("nan")
    cm = confusion_matrix(y_true, y_pred)
    ece, _, _, _ = expected_calibration_error(y_prob, y_true)
    return {
        "loss": total_loss / n,
        "acc": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "precision_mac": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_mac": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_mac": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "auc": auc,
        "mcc": matthews_corrcoef(y_true, y_pred),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "ece": ece,
        "cm": cm,
        "report": classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0),
    }


def run_experiment(dataset_name: str, train_root: Path, test_root: Path,
                    folder_to_class: Dict[str, str],
                    model_builder=None,
                    total_epochs: Optional[int] = None,
                    dropout: Optional[float] = None,
                    weight_decay: Optional[float] = None,
                    mixup_alpha: float = 0.2,
                    ema_decay: float = 0.999,
                    model_kwargs: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Trains a model built by `model_builder(dropout=..., **model_kwargs)`
    (defaults to build_smgf_mscaeca -- the full SMGF-Net + MS-CA-ECA model)
    with the SAME dataset loading / splitting / leakage-check / preprocessing
    / augmentation / optimizer / LR-schedule / CAWL+ loss / Mixup / EMA /
    flip-TTA pipeline every time. Used for the full model (§14) AND every
    component/internal ablation variant (§15/§16) -- this identical function
    call is what keeps the ablation protocol fair (section 5 of the task
    brief): same splits, same preprocessing, same augmentation, same image
    size, same batch size, same optimizer, same LR schedule, same epochs,
    same loss, same early stopping, same seed."""
    model_builder = model_builder or build_smgf_mscaeca
    total_epochs = TOTAL_EPOCHS if total_epochs is None else total_epochs
    dropout = 0.4 if dropout is None else dropout
    weight_decay = WEIGHT_DECAY * 1.5 if weight_decay is None else weight_decay
    model_kwargs = model_kwargs or {}

    print("\n" + "#" * 65)
    print(f"# EXPERIMENT: {dataset_name}  "
          f"[dropout={dropout} weight_decay={weight_decay:.2e} mixup_alpha={mixup_alpha} "
          f"ema_decay={ema_decay} epochs={total_epochs} model_kwargs={model_kwargs}]")
    print("#" * 65)

    all_train = scan_dataset_generic(train_root, folder_to_class, "train")
    test_s    = scan_dataset_generic(test_root, folder_to_class, "test")
    if len(all_train) == 0 or len(test_s) == 0:
        raise RuntimeError(f"[{dataset_name}] No images found — check paths.")
    train_s, val_s = train_test_split(
        all_train, test_size=0.15, random_state=SEED,
        stratify=[s["label"] for s in all_train])

    splits = dedupe_splits({"train": train_s, "val": val_s, "test": test_s})
    train_s, val_s, test_s = splits["train"], splits["val"], splits["test"]

    train_ds = BrainTumorDataset(train_s, is_train=True, augmenter=train_augment)
    val_ds   = BrainTumorDataset(val_s,   is_train=False)
    test_ds  = BrainTumorDataset(test_s,  is_train=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)

    if len(train_s) > len(val_s):
        train_eval_s, _ = train_test_split(
            train_s, train_size=len(val_s), random_state=SEED,
            stratify=[s["label"] for s in train_s])
    else:
        train_eval_s = train_s
    train_eval_ds = BrainTumorDataset(train_eval_s, is_train=False)
    train_eval_loader = DataLoader(train_eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True)

    model = model_builder(dropout=dropout, **model_kwargs)
    total_params = sum(p.numel() for p in model.parameters())
    ema = ModelEMA(model, decay=ema_decay)

    class_weights = compute_class_weights(train_s).to(DEVICE)
    criterion = CAWLPlus(class_weights, num_classes=NUM_CLASSES,
                          emb_dim=model.head.fc1.out_features).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, weight_decay=weight_decay)
    scaler = GradScaler() if USE_AMP else None

    history = {"epoch": [], "train_loss": [], "train_acc": [], "train_loss_aug": [], "train_acc_aug": [],
               "val_loss": [], "val_acc": [], "val_auc": [], "val_ece": [], "lr": []}

    _t0 = evaluate_v2(model, train_eval_loader, criterion)
    _v0 = evaluate_v2(model, val_loader, criterion)
    print(f"  epoch 000 (pre-training baseline)  "
          f"train_acc(clean)={_t0['acc']:.4f}  val_acc={_v0['acc']:.4f}  val_auc={_v0['auc']:.4f}")
    history["epoch"].append(0)
    history["train_loss"].append(_t0["loss"]); history["train_acc"].append(_t0["acc"])
    history["train_loss_aug"].append(_t0["loss"]); history["train_acc_aug"].append(_t0["acc"])
    history["val_loss"].append(_v0["loss"]); history["val_acc"].append(_v0["acc"])
    history["val_auc"].append(_v0["auc"]); history["val_ece"].append(_v0["ece"])
    history["lr"].append(0.0)

    best_val_auc = -1.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, total_epochs + 1):
        for g in optimizer.param_groups:
            g["lr"] = lr_at_epoch_v2(epoch)
        criterion.current_epoch = epoch

        train_loss, train_acc = train_one_epoch_mixup(
            model, train_loader, optimizer, criterion, scaler, ema=ema, mixup_alpha=mixup_alpha)
        train_clean = evaluate_v2(model, train_eval_loader, criterion)
        val_metrics = evaluate_v2(model, val_loader, criterion)

        print(f"  epoch {epoch:03d}  lr={lr_at_epoch_v2(epoch):.2e}  "
              f"train_acc(clean)={train_clean['acc']:.4f} train_acc(mixup)={train_acc:.4f}  "
              f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['acc']:.4f} "
              f"val_auc={val_metrics['auc']:.4f} val_ece={val_metrics['ece']:.4f}")

        history["epoch"].append(epoch)
        history["train_loss"].append(train_clean["loss"]); history["train_acc"].append(train_clean["acc"])
        history["train_loss_aug"].append(train_loss); history["train_acc_aug"].append(train_acc)
        history["val_loss"].append(val_metrics["loss"]); history["val_acc"].append(val_metrics["acc"])
        history["val_auc"].append(val_metrics["auc"]); history["val_ece"].append(val_metrics["ece"])
        history["lr"].append(lr_at_epoch_v2(epoch))

        if val_metrics["auc"] > best_val_auc:
            best_val_auc = val_metrics["auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f"  [EARLY STOP] no val AUC improvement for {EARLY_STOP_PATIENCE} epochs.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    raw_val = evaluate_v2(model, val_loader, criterion)

    ema_model = model_builder(dropout=dropout, **model_kwargs)
    ema.copy_to(ema_model)
    ema_val = evaluate_v2(ema_model, val_loader, criterion)

    if ema_val["auc"] >= raw_val["auc"]:
        print(f"  [FINAL WEIGHTS] EMA (val_auc={ema_val['auc']:.4f}) >= "
              f"best-checkpoint (val_auc={raw_val['auc']:.4f}) -> using EMA weights.")
        model = ema_model
    else:
        print(f"  [FINAL WEIGHTS] best-checkpoint (val_auc={raw_val['auc']:.4f}) > "
              f"EMA (val_auc={ema_val['auc']:.4f}) -> keeping best-checkpoint weights.")
        del ema_model

    test_metrics = evaluate_v2_tta(model, test_loader, criterion)
    print(f"[{dataset_name} [TEST, flip-TTA]] Accuracy={test_metrics['acc']:.4f}  "
          f"BalancedAcc={test_metrics['balanced_acc']:.4f}  F1(macro)={test_metrics['f1_mac']:.4f}  "
          f"AUC={test_metrics['auc']:.4f}  MCC={test_metrics['mcc']:.4f}  "
          f"CohenKappa={test_metrics['kappa']:.4f}  ECE={test_metrics['ece']:.4f}")
    print(test_metrics["report"])
    plot_confusion_matrix(test_metrics["cm"], title=f"{dataset_name} Confusion Matrix (flip-TTA)",
                           save_path=str(OUT_DIR / f"{dataset_name}_confusion_matrix.png"))

    torch.save(model.state_dict(), OUT_DIR / f"{dataset_name}.pth")

    return {
        "name": dataset_name, "model": model, "criterion": criterion,
        "total_params": total_params,
        "n_train": len(train_s), "n_val": len(val_s), "n_test": len(test_s),
        "train_samples": train_s, "val_samples": val_s, "test_samples": test_s,
        "test_metrics": test_metrics,
        "history": history,
        "config": {"dropout": dropout, "weight_decay": weight_decay, "mixup_alpha": mixup_alpha,
                   "ema_decay": ema_decay, "epochs": total_epochs, **SMGF_MSCAECA_CONFIG, **model_kwargs},
    }

print("run_experiment() ready (Mixup + EMA + flip-TTA, model-builder-generic).")


In [ ]:
def run_external_validation(model: nn.Module, criterion: CAWLPlus, dataset_name: str,
                             external_root: Path, folder_to_class: Dict[str, str],
                             known_splits: Optional[Dict[str, List[Dict]]] = None,
                             batch_size: int = BATCH_SIZE) -> Dict[str, Any]:
    """Evaluate an already-trained model on an external (out-of-distribution) dataset.
    known_splits: pass {"train": full_result["train_samples"], "val": full_result["val_samples"],
    "test": full_result["test_samples"]} to flag any file-hash overlap with the external set
    (cross-dataset leakage/contamination check)."""
    ext_samples = scan_dataset_generic(external_root, folder_to_class, dataset_name)
    if len(ext_samples) == 0:
        raise RuntimeError(f"[{dataset_name}] No images found under {external_root}")

    if known_splits is not None:
        known_hashes = set()
        for split_name, split_samples in known_splits.items():
            for s in split_samples:
                known_hashes.add(_file_hash(s["path"]))
        overlap = sum(1 for s in ext_samples if _file_hash(s["path"]) in known_hashes)
        if overlap:
            print(f"  [WARNING] {overlap}/{len(ext_samples)} external images are byte-identical "
                  f"to images already seen in train/val/test -- this dataset is CONTAMINATED, "
                  f"not independent external validation.")
        else:
            print(f"  [OK] no file-hash overlap between '{dataset_name}' and train/val/test.")

    ext_ds = BrainTumorDataset(ext_samples, is_train=False)
    ext_loader = DataLoader(ext_ds, batch_size=batch_size, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

    ext_metrics = evaluate_v2(model, ext_loader, criterion)
    print(f"\n[{dataset_name} EXTERNAL VALIDATION]  n={len(ext_samples)}  "
          f"Accuracy={ext_metrics['acc']:.4f}  BalancedAcc={ext_metrics['balanced_acc']:.4f}  "
          f"MacroPrec={ext_metrics['precision_mac']:.4f}  MacroRec={ext_metrics['recall_mac']:.4f}  "
          f"F1(macro)={ext_metrics['f1_mac']:.4f}  F1(weighted)={ext_metrics['f1_weighted']:.4f}  "
          f"AUC={ext_metrics['auc']:.4f}  MCC={ext_metrics['mcc']:.4f}  ECE={ext_metrics['ece']:.4f}")
    print(ext_metrics["report"])
    plot_confusion_matrix(ext_metrics["cm"], title=f"{dataset_name} \u2014 External Validation Confusion Matrix",
                           save_path=str(OUT_DIR / f"{dataset_name}_external_confusion_matrix.png"))
    return ext_metrics

print("run_external_validation() ready.")


In [ ]:
def denormalize_to_rgb01(img_tensor: torch.Tensor) -> np.ndarray:
    """[3,H,W] normalized tensor -> [H,W,3] float array in [0,1] for display/overlay."""
    mean = torch.tensor(IMAGENET_MEAN, device=img_tensor.device).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  device=img_tensor.device).view(3, 1, 1)
    img = (img_tensor.detach().cpu() * std.cpu() + mean.cpu()).clamp(0, 1)
    return img.permute(1, 2, 0).numpy()


def overlay_heatmap(rgb01: np.ndarray, cam01: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """cam01: 2D array in [0,1], same H,W as rgb01. Returns [H,W,3] uint8 overlay."""
    heat = cv2.applyColorMap((cam01 * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    base = rgb01.astype(np.float32)
    blended = (1 - alpha) * base + alpha * heat
    return (np.clip(blended, 0, 1) * 255).astype(np.uint8)


def load_input_tensor(path: str) -> torch.Tensor:
    """Single image -> [1,3,224,224] tensor on DEVICE, ready for model(...)."""
    return preprocess(path).unsqueeze(0).to(DEVICE)


class GradCAMPlusPlus:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.activations = None
        self.gradients = None
        self._fwd = target_layer.register_forward_hook(self._save_act)
        self._bwd = target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, module, inp, out):
        self.activations = out

    def _save_grad(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def remove(self):
        self._fwd.remove(); self._bwd.remove()

    def generate(self, input_tensor: torch.Tensor, class_idx: Optional[int] = None):
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        out = self.model(input_tensor)
        logits = out["logits"]
        if class_idx is None:
            class_idx = int(logits.argmax(dim=1).item())
        score = logits[:, class_idx].sum()
        score.backward()

        acts, grads = self.activations, self.gradients
        eps = 1e-8
        g2, g3 = grads ** 2, grads ** 3
        sum_acts = acts.sum(dim=(2, 3), keepdim=True)
        alpha_denom = 2 * g2 + sum_acts * g3
        alpha_denom = torch.where(alpha_denom.abs() > eps, alpha_denom,
                                   torch.full_like(alpha_denom, eps))
        alpha = g2 / alpha_denom
        weights = (alpha * F.relu(grads)).sum(dim=(2, 3), keepdim=True)

        cam = F.relu((weights * acts).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + eps)
        return cam, class_idx


class ScoreCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.activations = None
        self._fwd = target_layer.register_forward_hook(self._save_act)

    def _save_act(self, module, inp, out):
        self.activations = out

    def remove(self):
        self._fwd.remove()

    @torch.no_grad()
    def generate(self, input_tensor: torch.Tensor, class_idx: Optional[int] = None,
                 batch_size: int = 16):
        self.model.eval()
        out0 = self.model(input_tensor)
        if class_idx is None:
            class_idx = int(out0["logits"].argmax(dim=1).item())

        acts = self.activations[0]
        C = acts.shape[0]
        img_hw = input_tensor.shape[-2:]
        acts_up = F.interpolate(acts.unsqueeze(0), size=img_hw,
                                 mode="bilinear", align_corners=False).squeeze(0)

        eps = 1e-8
        c_min = acts_up.amin(dim=(1, 2), keepdim=True)
        c_max = acts_up.amax(dim=(1, 2), keepdim=True)
        masks = (acts_up - c_min) / (c_max - c_min + eps)

        base = input_tensor.expand(1, -1, -1, -1)
        scores = torch.zeros(C, device=input_tensor.device)
        for start in range(0, C, batch_size):
            chunk_masks = masks[start:start + batch_size].unsqueeze(1)
            masked_inputs = base * chunk_masks
            out = self.model(masked_inputs)
            probs = torch.softmax(out["logits"], dim=1)[:, class_idx]
            scores[start:start + chunk_masks.shape[0]] = probs

        weights = torch.softmax(scores, dim=0)
        cam = F.relu((weights.view(C, 1, 1) * acts_up).sum(dim=0))
        cam = cam.cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + eps)
        return cam, class_idx


from skimage.segmentation import slic
from sklearn.linear_model import Ridge

class LimeImageExplainer:
    def __init__(self, model: nn.Module, num_samples: int = 200, num_segments: int = 50,
                 kernel_width: float = 0.25, batch_size: int = 32):
        self.model = model
        self.num_samples = num_samples
        self.num_segments = num_segments
        self.kernel_width = kernel_width
        self.batch_size = batch_size

    @torch.no_grad()
    def explain(self, input_tensor: torch.Tensor, class_idx: Optional[int] = None,
                num_top_segments: int = 8, seed: int = SEED):
        self.model.eval()
        device = input_tensor.device
        mean = torch.tensor(IMAGENET_MEAN, device=device).view(3, 1, 1)
        std  = torch.tensor(IMAGENET_STD,  device=device).view(3, 1, 1)

        img_denorm = (input_tensor[0] * std + mean).clamp(0, 1)
        img_np = img_denorm.permute(1, 2, 0).cpu().numpy()
        segments = slic(img_np, n_segments=self.num_segments, compactness=10, start_label=0)
        n_segs = int(segments.max()) + 1
        segments_t = torch.from_numpy(segments).to(device)

        if class_idx is None:
            out0 = self.model(input_tensor)
            class_idx = int(out0["logits"].argmax(dim=1).item())

        baseline = img_denorm.mean(dim=(1, 2), keepdim=True).expand_as(img_denorm)

        rng = np.random.default_rng(seed)
        Z = rng.integers(0, 2, size=(self.num_samples, n_segs)).astype(np.float32)
        Z[0, :] = 1.0

        preds = np.zeros(self.num_samples, dtype=np.float32)
        for start in range(0, self.num_samples, self.batch_size):
            chunk = Z[start:start + self.batch_size]
            batch_imgs = []
            for row in chunk:
                on_segments = torch.tensor(np.where(row == 1)[0], device=device)
                mask = torch.isin(segments_t, on_segments).float().unsqueeze(0)
                perturbed = img_denorm * mask + baseline * (1 - mask)
                batch_imgs.append((perturbed - mean) / std)
            batch_imgs = torch.stack(batch_imgs, dim=0)
            out = self.model(batch_imgs)
            probs = torch.softmax(out["logits"], dim=1)[:, class_idx]
            preds[start:start + chunk.shape[0]] = probs.cpu().numpy()

        distances = np.sqrt(((1 - Z) ** 2).sum(axis=1)) / np.sqrt(n_segs)
        weights = np.exp(-(distances ** 2) / (self.kernel_width ** 2))

        surrogate = Ridge(alpha=1.0)
        surrogate.fit(Z, preds, sample_weight=weights)
        importances = surrogate.coef_

        top_segments = np.argsort(np.abs(importances))[::-1][:num_top_segments]
        return {"segments": segments, "importances": importances,
                "top_segments": top_segments, "class_idx": class_idx}

    @staticmethod
    def visualize(rgb01: np.ndarray, explanation: Dict[str, Any], positive_only: bool = True) -> np.ndarray:
        segments, importances = explanation["segments"], explanation["importances"]
        top = explanation["top_segments"]
        mask = np.zeros(segments.shape, dtype=bool)
        overlay = (rgb01.copy() * 255).astype(np.uint8)
        for seg_id in top:
            if positive_only and importances[seg_id] <= 0:
                continue
            seg_mask = segments == seg_id
            mask |= seg_mask
            boundary = cv2.dilate(seg_mask.astype(np.uint8), np.ones((3, 3), np.uint8)) - seg_mask.astype(np.uint8)
            overlay[boundary.astype(bool)] = [255, 255, 0]
        dim = overlay.copy()
        dim[~mask] = (dim[~mask] * 0.35).astype(np.uint8)
        return dim


def explain_sample(model: nn.Module, image_path: str, true_label: Optional[str] = None,
                    save_path: Optional[str] = None):
    """Grad-CAM++ / Score-CAM / LIME, all targeted at model.attn (MS-CA-ECA) --
    the attribution/localization terminology below deliberately avoids any
    claim of clinical correctness."""
    input_tensor = load_input_tensor(image_path)
    rgb01 = denormalize_to_rgb01(input_tensor[0])

    gcpp = GradCAMPlusPlus(model, model.attn)
    cam_gcpp, pred_idx = gcpp.generate(input_tensor)
    gcpp.remove()

    scam = ScoreCAM(model, model.attn)
    cam_score, _ = scam.generate(input_tensor, class_idx=pred_idx)
    scam.remove()

    lime_exp = LimeImageExplainer(model).explain(input_tensor, class_idx=pred_idx)

    pred_name = CLASS_NAMES[pred_idx]
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    axes[0].imshow(rgb01); axes[0].set_title(f"Input\n(true: {true_label or '?'})")
    axes[1].imshow(overlay_heatmap(rgb01, cam_gcpp)); axes[1].set_title(f"Grad-CAM++\n(pred: {pred_name})")
    axes[2].imshow(overlay_heatmap(rgb01, cam_score)); axes[2].set_title("Score-CAM")
    axes[3].imshow(LimeImageExplainer.visualize(rgb01, lime_exp)); axes[3].set_title("LIME")
    for ax in axes: ax.axis("off")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return {"pred_idx": pred_idx, "pred_name": pred_name,
            "gradcampp": cam_gcpp, "scorecam": cam_score, "lime": lime_exp}

print("XAI utilities ready (Grad-CAM++, Score-CAM, LIME), targeted at MS-CA-ECA.")


In [ ]:
def _gaussian_blur_baseline(img_denorm: torch.Tensor, ksize: int = 31) -> torch.Tensor:
    """Blurred version of the image, used as the 'nothing here' baseline for
    deletion/insertion -- less of an artificial edge artifact than a flat/zero fill."""
    img_np = (img_denorm.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    blurred = cv2.GaussianBlur(img_np, (ksize, ksize), 0)
    return torch.from_numpy(blurred).float().permute(2, 0, 1) / 255.0


@torch.no_grad()
def deletion_insertion_pixelwise(model: nn.Module, input_tensor: torch.Tensor, cam01: np.ndarray,
                                  class_idx: int, n_steps: int = 20) -> Dict[str, Any]:
    """For a continuous per-pixel heatmap (Grad-CAM++ or Score-CAM). Ranks pixels by
    cam value, then reveals/removes them n_steps at a time. Returns probability
    curves and their AUC (trapezoidal, x-axis = fraction of pixels processed)."""
    device = input_tensor.device
    mean = torch.tensor(IMAGENET_MEAN, device=device).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  device=device).view(3, 1, 1)
    img_denorm = (input_tensor[0] * std + mean).clamp(0, 1)
    baseline = _gaussian_blur_baseline(img_denorm).to(device)

    H, W = cam01.shape
    order = np.argsort(cam01.flatten())[::-1]
    n_pixels = H * W
    step_size = max(1, n_pixels // n_steps)
    steps = [min(i * step_size, n_pixels) for i in range(n_steps + 1)]

    del_imgs = []
    flat_mask = np.zeros(n_pixels, dtype=bool)
    for k in steps:
        flat_mask[:] = False
        flat_mask[order[:k]] = True
        m = torch.from_numpy(flat_mask.reshape(H, W)).to(device).float().unsqueeze(0)
        perturbed = baseline * m + img_denorm * (1 - m)
        del_imgs.append((perturbed - mean) / std)
    del_batch = torch.stack(del_imgs, dim=0)
    del_out = model(del_batch)
    del_probs = torch.softmax(del_out["logits"], dim=1)[:, class_idx].cpu().numpy()

    ins_imgs = []
    for k in steps:
        flat_mask[:] = False
        flat_mask[order[:k]] = True
        m = torch.from_numpy(flat_mask.reshape(H, W)).to(device).float().unsqueeze(0)
        perturbed = img_denorm * m + baseline * (1 - m)
        ins_imgs.append((perturbed - mean) / std)
    ins_batch = torch.stack(ins_imgs, dim=0)
    ins_out = model(ins_batch)
    ins_probs = torch.softmax(ins_out["logits"], dim=1)[:, class_idx].cpu().numpy()

    x = np.array(steps) / n_pixels
    del_auc = float(np.trapz(del_probs, x))
    ins_auc = float(np.trapz(ins_probs, x))
    return {"x": x, "deletion_probs": del_probs, "insertion_probs": ins_probs,
            "deletion_auc": del_auc, "insertion_auc": ins_auc}


@torch.no_grad()
def deletion_insertion_segments(model: nn.Module, input_tensor: torch.Tensor,
                                 explanation: Dict[str, Any], class_idx: int) -> Dict[str, Any]:
    """Same protocol for LIME's segment-based explanation -- ranks superpixels by
    |importance| and reveals/removes them one at a time (x-axis = fraction of area)."""
    device = input_tensor.device
    mean = torch.tensor(IMAGENET_MEAN, device=device).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  device=device).view(3, 1, 1)
    img_denorm = (input_tensor[0] * std + mean).clamp(0, 1)
    baseline = _gaussian_blur_baseline(img_denorm).to(device)

    segments, importances = explanation["segments"], explanation["importances"]
    ranked = np.argsort(np.abs(importances))[::-1]
    total_area = segments.size

    del_imgs, ins_imgs, areas = [], [], []
    for n_segs_used in range(0, len(ranked) + 1):
        used = ranked[:n_segs_used]
        seg_mask = np.isin(segments, used)
        areas.append(seg_mask.sum() / total_area)
        m = torch.from_numpy(seg_mask).to(device).float().unsqueeze(0)
        del_perturbed = baseline * m + img_denorm * (1 - m)
        ins_perturbed = img_denorm * m + baseline * (1 - m)
        del_imgs.append((del_perturbed - mean) / std)
        ins_imgs.append((ins_perturbed - mean) / std)

    del_out = model(torch.stack(del_imgs, dim=0))
    ins_out = model(torch.stack(ins_imgs, dim=0))
    del_probs = torch.softmax(del_out["logits"], dim=1)[:, class_idx].cpu().numpy()
    ins_probs = torch.softmax(ins_out["logits"], dim=1)[:, class_idx].cpu().numpy()

    x = np.array(areas)
    del_auc = float(np.trapz(del_probs, x))
    ins_auc = float(np.trapz(ins_probs, x))
    return {"x": x, "deletion_probs": del_probs, "insertion_probs": ins_probs,
            "deletion_auc": del_auc, "insertion_auc": ins_auc}


def run_xai_batch_validation(model: nn.Module, samples: List[Dict], n_per_class: int = 5,
                              seed: int = SEED, save_grid_path: Optional[str] = None) -> Dict[str, Any]:
    """Picks n_per_class samples per class, runs Grad-CAM++/Score-CAM/LIME on each
    (targeted at model.attn), scores faithfulness (deletion/insertion AUC), and saves
    a comparison grid. Returns per-method aggregate stats (mean +/- std)."""
    rng = random.Random(seed)
    by_class: Dict[str, List[Dict]] = {c: [] for c in CLASS_NAMES}
    for s in samples:
        by_class[s["class_name"]].append(s)
    chosen = []
    for c, lst in by_class.items():
        chosen.extend(rng.sample(lst, min(n_per_class, len(lst))))

    rows = []
    grid_imgs = []

    for s in chosen:
        input_tensor = load_input_tensor(s["path"])
        rgb01 = denormalize_to_rgb01(input_tensor[0])

        gcpp = GradCAMPlusPlus(model, model.attn)
        cam_gcpp, pred_idx = gcpp.generate(input_tensor)
        gcpp.remove()
        di_gcpp = deletion_insertion_pixelwise(model, input_tensor, cam_gcpp, pred_idx)

        scam = ScoreCAM(model, model.attn)
        cam_score, _ = scam.generate(input_tensor, class_idx=pred_idx)
        scam.remove()
        di_score = deletion_insertion_pixelwise(model, input_tensor, cam_score, pred_idx)

        lime_exp = LimeImageExplainer(model).explain(input_tensor, class_idx=pred_idx)
        di_lime = deletion_insertion_segments(model, input_tensor, lime_exp, pred_idx)

        correct = (pred_idx == CLASS_TO_IDX[s["class_name"]])
        rows.append({
            "path": s["path"], "true_class": s["class_name"], "pred_class": CLASS_NAMES[pred_idx],
            "correct": correct,
            "gradcampp_del_auc": di_gcpp["deletion_auc"], "gradcampp_ins_auc": di_gcpp["insertion_auc"],
            "scorecam_del_auc": di_score["deletion_auc"], "scorecam_ins_auc": di_score["insertion_auc"],
            "lime_del_auc": di_lime["deletion_auc"], "lime_ins_auc": di_lime["insertion_auc"],
        })
        grid_imgs.append((rgb01, overlay_heatmap(rgb01, cam_gcpp), overlay_heatmap(rgb01, cam_score),
                           LimeImageExplainer.visualize(rgb01, lime_exp),
                           f"{s['class_name']}\n(pred: {CLASS_NAMES[pred_idx]})"))

    df = pd.DataFrame(rows)
    summary = {}
    for method in ["gradcampp", "scorecam", "lime"]:
        summary[method] = {
            "deletion_auc_mean": df[f"{method}_del_auc"].mean(), "deletion_auc_std": df[f"{method}_del_auc"].std(),
            "insertion_auc_mean": df[f"{method}_ins_auc"].mean(), "insertion_auc_std": df[f"{method}_ins_auc"].std(),
        }
    print("\n[XAI FAITHFULNESS SUMMARY]  (lower deletion-AUC = better, higher insertion-AUC = better)")
    for method, m in summary.items():
        print(f"  {method:10s}  deletion={m['deletion_auc_mean']:.4f}\u00b1{m['deletion_auc_std']:.4f}  "
              f"insertion={m['insertion_auc_mean']:.4f}\u00b1{m['insertion_auc_std']:.4f}")

    n = len(grid_imgs)
    fig, axes = plt.subplots(n, 4, figsize=(14, 3.2 * n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for row, (rgb01, gcpp_img, score_img, lime_img, title) in enumerate(grid_imgs):
        axes[row, 0].imshow(rgb01); axes[row, 0].set_ylabel(title, fontsize=9, rotation=0,
                                                              labelpad=45, va="center")
        axes[row, 1].imshow(gcpp_img); axes[row, 2].imshow(score_img); axes[row, 3].imshow(lime_img)
        for col in range(4): axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
    for col, name in enumerate(["Input", "Grad-CAM++", "Score-CAM", "LIME"]):
        axes[0, col].set_title(name)
    plt.tight_layout()
    if save_grid_path: plt.savefig(save_grid_path, dpi=150, bbox_inches="tight")
    plt.show()

    return {"per_sample": df, "summary": summary}





In [ ]:
def compute_method_agreement(cam_a: np.ndarray, cam_b: np.ndarray, top_percent: float = 0.15) -> Dict[str, float]:
    """Agreement between two continuous heatmaps of the same image: IoU of their
    top-k% most-important regions, plus Spearman rank correlation over all pixels."""
    assert cam_a.shape == cam_b.shape
    n = cam_a.size
    k = max(1, int(n * top_percent))

    thresh_a = np.partition(cam_a.flatten(), -k)[-k]
    thresh_b = np.partition(cam_b.flatten(), -k)[-k]
    mask_a = cam_a >= thresh_a
    mask_b = cam_b >= thresh_b
    intersection = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    iou = float(intersection / union) if union > 0 else 0.0

    rho, _ = spearmanr(cam_a.flatten(), cam_b.flatten())
    return {"iou_top_k": iou, "spearman_rho": float(rho)}


def lime_segments_to_dense_map(explanation: Dict[str, Any]) -> np.ndarray:
    """Converts LIME's per-segment importances into a dense [H,W] map so it can be
    compared against Grad-CAM++/Score-CAM with the same IoU/correlation function."""
    segments, importances = explanation["segments"], explanation["importances"]
    dense = np.zeros(segments.shape, dtype=np.float32)
    for seg_id, imp in enumerate(importances):
        dense[segments == seg_id] = imp
    dense = np.clip(dense, 0, None)
    if dense.max() > 0:
        dense = dense / dense.max()
    return dense


def run_agreement_analysis(model: nn.Module, samples: List[Dict], n_samples: int = 20,
                            seed: int = SEED) -> pd.DataFrame:
    rng = random.Random(seed)
    chosen = rng.sample(samples, min(n_samples, len(samples)))
    rows = []
    for s in chosen:
        input_tensor = load_input_tensor(s["path"])

        gcpp = GradCAMPlusPlus(model, model.attn)
        cam_gcpp, pred_idx = gcpp.generate(input_tensor)
        gcpp.remove()

        scam = ScoreCAM(model, model.attn)
        cam_score, _ = scam.generate(input_tensor, class_idx=pred_idx)
        scam.remove()

        lime_exp = LimeImageExplainer(model).explain(input_tensor, class_idx=pred_idx)
        cam_lime = lime_segments_to_dense_map(lime_exp)

        gs = compute_method_agreement(cam_gcpp, cam_score)
        gl = compute_method_agreement(cam_gcpp, cam_lime)
        sl = compute_method_agreement(cam_score, cam_lime)
        rows.append({
            "path": s["path"], "true_class": s["class_name"], "pred_class": CLASS_NAMES[pred_idx],
            "gradcampp_vs_scorecam_iou": gs["iou_top_k"], "gradcampp_vs_scorecam_rho": gs["spearman_rho"],
            "gradcampp_vs_lime_iou": gl["iou_top_k"], "gradcampp_vs_lime_rho": gl["spearman_rho"],
            "scorecam_vs_lime_iou": sl["iou_top_k"], "scorecam_vs_lime_rho": sl["spearman_rho"],
        })
    df = pd.DataFrame(rows)
    print("\n[CROSS-METHOD AGREEMENT]  (mean over {} samples)".format(len(df)))
    for pair in ["gradcampp_vs_scorecam", "gradcampp_vs_lime", "scorecam_vs_lime"]:
        print(f"  {pair:24s}  IoU={df[f'{pair}_iou'].mean():.3f}  "
              f"Spearman \u03c1={df[f'{pair}_rho'].mean():.3f}")
    return df


# output-side -> input-side order, using SMGFNetMSCAECA's own submodule names
CASCADE_BLOCKS = ["head", "attn", "fusion", "morphology", "multiscale", "backbone"]

def _randomize_block_(module: nn.Module, generator: Optional[torch.Generator] = None):
    """Reinitializes a block's weights with random noise matched to each parameter's
    existing scale, for a genuine (not just magnitude-changing) randomization."""
    with torch.no_grad():
        for p in module.parameters():
            std = p.data.std().clamp(min=1e-3)
            noise = torch.randn(p.shape, generator=generator, device="cpu").to(p.device) \
                if generator is not None else torch.randn(p.shape, device=p.device)
            p.data.copy_(noise * std)


def run_randomization_sanity_check(model: nn.Module, sample_path: str) -> Dict[str, Any]:
    """Cascading randomization test (Adebayo et al., 2018): progressively randomize
    the model's weights output-to-input, recomputing Grad-CAM++ each time. A real,
    weight-dependent explanation should degrade as more of the trained model is
    destroyed; if agreement stays high, Grad-CAM++ is acting like an edge detector
    on this input rather than explaining the trained model."""
    import copy
    input_tensor = load_input_tensor(sample_path)

    model_copy = copy.deepcopy(model)
    gcpp = GradCAMPlusPlus(model_copy, model_copy.attn)
    cam_original, class_idx = gcpp.generate(input_tensor)
    gcpp.remove()

    generator = torch.Generator(device="cpu").manual_seed(SEED)
    results = [{"stage": "original (trained weights)", "spearman_rho": 1.0, "iou_top_k": 1.0}]
    print("[SANITY CHECK] cascading randomization vs. Grad-CAM++ agreement:")
    for block_name in CASCADE_BLOCKS:
        _randomize_block_(getattr(model_copy, block_name), generator)
        gcpp = GradCAMPlusPlus(model_copy, model_copy.attn)
        cam_rand, _ = gcpp.generate(input_tensor, class_idx=class_idx)
        gcpp.remove()
        agreement = compute_method_agreement(cam_original, cam_rand)
        results.append({"stage": f"+ {block_name} randomized", **agreement})
        print(f"  after randomizing '{block_name}':  IoU={agreement['iou_top_k']:.3f}  "
              f"Spearman \u03c1={agreement['spearman_rho']:.3f}")

    del model_copy
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return {"class_idx": class_idx, "cascade": results}


def plot_randomization_cascade(model: nn.Module, sample_path: str, save_path: Optional[str] = None):
    """Visual companion to run_randomization_sanity_check: Grad-CAM++ heatmap at
    each cascade stage, side by side."""
    import copy
    input_tensor = load_input_tensor(sample_path)
    rgb01 = denormalize_to_rgb01(input_tensor[0])

    model_copy = copy.deepcopy(model)
    stage_names = ["original"] + CASCADE_BLOCKS
    cams = []
    generator = torch.Generator(device="cpu").manual_seed(SEED)
    gcpp = GradCAMPlusPlus(model_copy, model_copy.attn)
    cam, class_idx = gcpp.generate(input_tensor)
    gcpp.remove()
    cams.append(cam)
    for block_name in CASCADE_BLOCKS:
        _randomize_block_(getattr(model_copy, block_name), generator)
        gcpp = GradCAMPlusPlus(model_copy, model_copy.attn)
        cam, _ = gcpp.generate(input_tensor, class_idx=class_idx)
        gcpp.remove()
        cams.append(cam)

    fig, axes = plt.subplots(1, len(cams), figsize=(4 * len(cams), 4))
    for ax, cam, name in zip(axes, cams, stage_names):
        ax.imshow(overlay_heatmap(rgb01, cam)); ax.set_title(name, fontsize=9); ax.axis("off")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    del model_copy
    if DEVICE == "cuda":
        torch.cuda.empty_cache()





In [ ]:
def select_stratified_subset(samples: List[Dict], n_per_class: int, seed: int = SEED) -> List[Dict]:
    """Same per-class random sampling pattern already used by run_xai_batch_validation
    / run_agreement_analysis (§14) -- factored out here so it's reused for the test
    subset, the validation-threshold subset (§9), and the external-validation
    subset (§11) instead of re-implemented three times."""
    rng = random.Random(seed)
    by_class: Dict[str, List[Dict]] = {c: [] for c in CLASS_NAMES}
    for s in samples:
        by_class[s["class_name"]].append(s)
    chosen = []
    for c, lst in by_class.items():
        chosen.extend(rng.sample(lst, min(n_per_class, len(lst))))
    return chosen





## §2 · Data, checkpoint, helper functions

In [ ]:
import glob, hashlib
from scipy.ndimage import gaussian_filter
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score

CKPT_NAME = "A_full_smgfnet_mscaeca.pth"
CKPT_PATH = None          # set a full path here if you do have the file somewhere
TRAIN_IF_MISSING = True   # no checkpoint -> retrain the seed-42 model once (about as long as the stage-1 run) and save it
TRAIN_USE_AMEF   = False  # preprocessing used for that retraining. False = the stage-1 notebook's preprocess(), which produced the 96.28 % run.
_cands = [Path("/kaggle/working")/CKPT_NAME] + [Path(p) for p in glob.glob(f"/kaggle/input/**/{CKPT_NAME}", recursive=True)]
CKPT_PATH = Path(CKPT_PATH) if CKPT_PATH else next((p for p in _cands if p.exists()), None)

BRISC_FOLDER_TO_CLASS = {"glioma": "glioma", "meningioma": "meningioma", "no_tumor": "notumor", "pituitary": "pituitary"}
RETRAINED_HERE = False
if CKPT_PATH is None:
    assert TRAIN_IF_MISSING, f"{CKPT_NAME} not found and TRAIN_IF_MISSING is False"
    print("No checkpoint found -> retraining the seed-42 model (stage-1 recipe: Mixup, EMA, flip-TTA, 50 epochs).")
    USE_AMEF = TRAIN_USE_AMEF
    full_result = run_experiment("A_full_smgfnet_mscaeca", BRISC_TRAIN_ROOT, BRISC_TEST_ROOT, BRISC_FOLDER_TO_CLASS,
                                 model_builder=build_smgf_mscaeca)
    CKPT_PATH = Path(OUT_DIR) / CKPT_NAME          # run_experiment has just saved it
    RETRAINED_HERE = True
    print("saved:", CKPT_PATH, "  -> download it / add it to a Kaggle dataset so it is kept this time")
print("checkpoint:", CKPT_PATH, "| retrained in this session:", RETRAINED_HERE)

all_train = scan_dataset_generic(BRISC_TRAIN_ROOT, BRISC_FOLDER_TO_CLASS, "train")
test_s    = scan_dataset_generic(BRISC_TEST_ROOT,  BRISC_FOLDER_TO_CLASS, "test")
from sklearn.model_selection import train_test_split
train_s, val_s = train_test_split(all_train, test_size=0.15, random_state=SEED, stratify=[s["label"] for s in all_train])
_sp = dedupe_splits({"train": train_s, "val": val_s, "test": test_s})
train_s, val_s, test_s = _sp["train"], _sp["val"], _sp["test"]
print(f"train {len(train_s)}  val {len(val_s)}  test {len(test_s)}   (paper: 4,209 / 746 / 995)")
assert len(test_s) == 995, "test set differs from the paper's 995 images"

model = build_smgf_mscaeca(dropout=0.4).to(DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()
print("parameters:", sum(p.numel() for p in model.parameters()))
class_weights = compute_class_weights(train_s).to(DEVICE)
criterion = CAWLPlus(class_weights, num_classes=NUM_CLASSES, emb_dim=model.head.fc1.out_features).to(DEVICE)

@torch.no_grad()
def predict_samples(model, samples, tta=False, batch_size=BATCH_SIZE):
    loader = DataLoader(BrainTumorDataset(samples, is_train=False), batch_size=batch_size,
                        shuffle=False, num_workers=NUM_WORKERS)
    model.eval(); P = []
    for b in loader:
        img = b["image"].to(DEVICE)
        p = torch.softmax(model(img)["logits"], 1)
        if tta:
            p = (p + torch.softmax(model(torch.flip(img, dims=[3]))["logits"], 1)) / 2
        P.append(p.cpu().numpy())
    return np.concatenate(P)

def boot_ci(y, pred, fn, n_boot=1000, seed=42, alpha=0.05):
    rng = np.random.default_rng(seed); n = len(y); vals = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n); vals.append(fn(y[i], pred[i]))
    return float(fn(y, pred)), float(np.percentile(vals, 100*alpha/2)), float(np.percentile(vals, 100*(1-alpha/2)))

y_test = np.array([s["label"] for s in test_s])


## §3 · Which preprocessing reproduces the reported result?

The paper reports 95.28 % (single pass) and 96.28 % (flip-TTA) for this checkpoint. The stage-1 `preprocess()` does not apply AMEF,
while the federated notebooks do. This cell evaluates the checkpoint both ways and keeps the setting that reproduces the reported numbers.

In [ ]:
rows = []
flags = [USE_AMEF] if RETRAINED_HERE else [False, True]     # a model trained here has only one valid preprocessing
for flag in flags:
    USE_AMEF = flag
    p1 = predict_samples(model, test_s, tta=False); p2 = predict_samples(model, test_s, tta=True)
    rows.append({"USE_AMEF": flag, "acc_single_pass": accuracy_score(y_test, p1.argmax(1)),
                 "acc_flip_tta": accuracy_score(y_test, p2.argmax(1))})
amef_df = pd.DataFrame(rows); amef_df.to_csv(OUT_DIR / "amef_check.csv", index=False)
print(amef_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
amef_df["gap"] = (amef_df.acc_single_pass - 0.9528).abs() + (amef_df.acc_flip_tta - 0.9628).abs()
USE_AMEF = bool(amef_df.sort_values("gap").iloc[0]["USE_AMEF"])
print(f"\nUSE_AMEF = {USE_AMEF} is used for everything below.")
if RETRAINED_HERE:
    ok = amef_df["gap"].min() < 0.004
    print("RETRAINED MODEL reproduces 95.28 / 96.28:" , ok)
    if not ok:
        print("-> The analyses below describe THIS retrained model, not the one behind the paper's 96.28 %. "
              "Report its accuracy next to every number from this notebook.")
else:
    assert amef_df["gap"].min() < 0.004, "neither setting reproduces 95.28 / 96.28 -- check the checkpoint and the test set before continuing"
P_single = predict_samples(model, test_s, tta=False); P_tta = predict_samples(model, test_s, tta=True)
np.savez_compressed(OUT_DIR / "brisc_test_predictions_seed42.npz", y_true=y_test, prob_single=P_single, prob_tta=P_tta,
                    paths=np.array([s["path"] for s in test_s]))


## §4 · External cohorts: contaminated vs clean subsets, with bootstrap CIs (points 62 and 63)

An external image is flagged *contaminated* if its MD5 hash equals that of any image in the (de-duplicated) BRISC2025 train, validation, or test
set. Metrics are reported on all images, on the clean subset only, and on the contaminated subset only, with 95 % bootstrap CIs
(1,000 resamples of images). The paper's external numbers used single-pass inference, so that is the primary column; flip-TTA is given for reference.
Dataset 1 and Dataset 2 in the paper are the two cohorts with n = 1,311 and n = 889.

In [ ]:
_ds1_root = Path("/kaggle/input/datasets/neyamul/bt-mendeley-3")
_ds1_subdirs = [d for d in _ds1_root.iterdir() if d.is_dir()]
assert len(_ds1_subdirs) == 1, _ds1_subdirs
EXTERNAL_DATASETS = [
    {"name": "mendeley3", "root": _ds1_subdirs[0] / "Test",
     "folder_to_class": {"glioma_tumor": "glioma", "meningioma_tumor": "meningioma", "no_tumor": "notumor", "pituitary_tumor": "pituitary"}},
    {"name": "mendeley5", "root": Path("/kaggle/input/datasets/neyamul/bt-mendeley-5/Brain Tumor data/Testing"),
     "folder_to_class": {"glioma": "glioma", "meningioma": "meningioma", "notumor": "notumor", "pituitary": "pituitary"}},
    {"name": "mendeley_epic_cscr", "root": Path("/kaggle/input/datasets/neyamul/mendeley-bt-data/Epic and CSCR hospital Dataset/Test"),
     "folder_to_class": {"glioma": "glioma", "meningioma": "meningioma", "notumor": "notumor", "pituitary": "pituitary"}},
    {"name": "mendeley2", "root": Path("/kaggle/input/datasets/neyamul/mendeley-2/test"),
     "folder_to_class": {"glioma": "glioma", "meningioma": "meningioma", "no_tumor": "notumor", "pituitary": "pituitary"}},
]
known_hashes = {_file_hash(s["path"]) for part in (train_s, val_s, test_s) for s in part}
print("BRISC hashes:", len(known_hashes))

def _metrics(y, pr, prob):
    out = {"n": int(len(y)), "acc": accuracy_score(y, pr), "bal_acc": balanced_accuracy_score(y, pr),
           "macro_f1": f1_score(y, pr, average="macro", zero_division=0)}
    return out

ext_rows = []
for ds in EXTERNAL_DATASETS:
    if not ds["root"].exists():
        print(f"[skip] {ds['name']}: {ds['root']} not found"); continue
    smp = scan_dataset_generic(ds["root"], ds["folder_to_class"], ds["name"])
    y = np.array([s["label"] for s in smp])
    flag = np.array([_file_hash(s["path"]) in known_hashes for s in smp])
    Ps, Pt = predict_samples(model, smp, tta=False), predict_samples(model, smp, tta=True)
    np.savez_compressed(OUT_DIR / f"ext_predictions_{ds['name']}.npz", y_true=y, prob_single=Ps, prob_tta=Pt,
                        contaminated=flag, paths=np.array([s["path"] for s in smp]))
    print(f"\n[{ds['name']}] n={len(smp)}  contaminated={flag.sum()}  clean={(~flag).sum()}")
    for subset, m in [("all", np.ones(len(y), bool)), ("clean", ~flag), ("contaminated", flag)]:
        if m.sum() < 5: continue
        row = {"dataset": ds["name"], "subset": subset, "n": int(m.sum())}
        for tag, P in [("single", Ps), ("tta", Pt)]:
            pr = P[m].argmax(1)
            a, alo, ahi = boot_ci(y[m], pr, accuracy_score)
            f, flo, fhi = boot_ci(y[m], pr, lambda yy, pp: f1_score(yy, pp, average="macro", zero_division=0))
            b, blo, bhi = boot_ci(y[m], pr, balanced_accuracy_score)
            row.update({f"acc_{tag}": a, f"acc_{tag}_lo": alo, f"acc_{tag}_hi": ahi,
                        f"macro_f1_{tag}": f, f"macro_f1_{tag}_lo": flo, f"macro_f1_{tag}_hi": fhi,
                        f"bal_acc_{tag}": b, f"bal_acc_{tag}_lo": blo, f"bal_acc_{tag}_hi": bhi})
            try: row[f"auc_{tag}"] = roc_auc_score(y[m], P[m], multi_class="ovr", average="macro")
            except Exception: row[f"auc_{tag}"] = np.nan
        ext_rows.append(row)
ext_df = pd.DataFrame(ext_rows); ext_df.to_csv(OUT_DIR / "ext_subset_metrics.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
print(ext_df[["dataset","subset","n","acc_single","acc_single_lo","acc_single_hi","macro_f1_single","macro_f1_single_lo","macro_f1_single_hi","acc_tta"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))


## §5 · Faithfulness against random-saliency baselines (point 65)

Same 32 stratified test images as the paper (8 per class). For each image the deletion and insertion AUC of Grad-CAM++, Score-CAM and LIME
is compared with two random baselines: pixel-wise uniform noise, and Gaussian-smoothed noise (σ = 16 px) that has spatial coherence.
Each random baseline is averaged over 20 draws per image. `insertion − deletion` is the single score to read:
a random map should be near 0, and a faithful map should be clearly positive.

In [ ]:
EXPLAIN_N_PER_CLASS = 8
explain_subset = select_stratified_subset(test_s, EXPLAIN_N_PER_CLASS)
print("explained subset:", len(explain_subset), [sum(s["class_name"] == c for s in explain_subset) for c in CLASS_NAMES])
rng_rand = np.random.default_rng(SEED)
N_RAND = 20

def random_map(kind):
    z = rng_rand.random((IMG_SIZE, IMG_SIZE))
    if kind == "smooth": z = gaussian_filter(z, sigma=16)
    return (z - z.min()) / (z.max() - z.min() + 1e-8)

rows = []
for k, s in enumerate(explain_subset):
    x = load_input_tensor(s["path"])
    g = GradCAMPlusPlus(model, model.attn); cam_g, cls = g.generate(x); g.remove()
    sc = ScoreCAM(model, model.attn); cam_s, _ = sc.generate(x, class_idx=cls); sc.remove()
    di = {"gradcam++": deletion_insertion_pixelwise(model, x, cam_g, cls),
          "scorecam": deletion_insertion_pixelwise(model, x, cam_s, cls)}
    lime_exp = LimeImageExplainer(model).explain(x, class_idx=cls)
    di["lime"] = deletion_insertion_segments(model, x, lime_exp, cls)
    for kind in ["uniform", "smooth"]:
        r = [deletion_insertion_pixelwise(model, x, random_map(kind), cls) for _ in range(N_RAND)]
        di[f"random_{kind}"] = {"deletion_auc": float(np.mean([a["deletion_auc"] for a in r])),
                                "insertion_auc": float(np.mean([a["insertion_auc"] for a in r]))}
    for m, d in di.items():
        rows.append({"idx": k, "class": s["class_name"], "method": m, "deletion_auc": d["deletion_auc"],
                     "insertion_auc": d["insertion_auc"], "ins_minus_del": d["insertion_auc"] - d["deletion_auc"]})
    print(f"  {k+1}/{len(explain_subset)} done", end="\r")
faith_df = pd.DataFrame(rows); faith_df.to_csv(OUT_DIR / "xai_faithfulness_per_sample_vs_random.csv", index=False)
summ = faith_df.groupby("method")[["deletion_auc", "insertion_auc", "ins_minus_del"]].agg(["mean", "std"]).round(4)
print(summ.to_string())
# paired difference to the smooth-random baseline, bootstrap over images
piv = faith_df.pivot(index="idx", columns="method", values="ins_minus_del")
out = []
for m in ["gradcam++", "scorecam", "lime"]:
    for base in ["random_uniform", "random_smooth"]:
        d = (piv[m] - piv[base]).to_numpy(); r = np.random.default_rng(1)
        bs = [d[r.integers(0, len(d), len(d))].mean() for _ in range(2000)]
        out.append({"method": m, "baseline": base, "mean_diff": d.mean(), "ci_lo": np.percentile(bs, 2.5), "ci_hi": np.percentile(bs, 97.5)})
vs_df = pd.DataFrame(out); vs_df.to_csv(OUT_DIR / "xai_faithfulness_vs_random.csv", index=False)
print("\nInsertion-minus-deletion, method minus random baseline (paired, 95% bootstrap CI over images):")
print(vs_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))


## §6 · Weight-randomization check (point 66)

Cascading randomization (output → input, Adebayo et al. 2018) with three independent random seeds, for Grad-CAM++ on all 32 images and
Score-CAM on 8 of them (2 per class). Reported per stage: mean Spearman ρ and mean top-15 % IoU against the trained-model map.
Also reported: (a) each block randomized **alone**, and (b) a **chance level**: the agreement between the maps of two *different*
fully randomized models.

*Pass criterion, fixed before looking at results:* the check is passed if the mean Spearman ρ after the full cascade is below 0.2 and
within 0.1 of the chance level; otherwise the map is partly insensitive to the trained weights.

In [ ]:
import copy
SEEDS_RAND = [0, 1, 2]
sc_subset = [s for c in CLASS_NAMES for s in [x for x in explain_subset if x["class_name"] == c][:2]]
orig = {}   # path -> (input, cls, gcpp cam, scam cam or None)
for s in explain_subset:
    x = load_input_tensor(s["path"])
    g = GradCAMPlusPlus(model, model.attn); cam_g, cls = g.generate(x); g.remove()
    cam_s = None
    if s in sc_subset:
        sc = ScoreCAM(model, model.attn); cam_s, _ = sc.generate(x, class_idx=cls); sc.remove()
    orig[s["path"]] = (x, cls, cam_g, cam_s)

def cams_for(m, method):
    out = {}
    for s in (explain_subset if method == "gradcam++" else sc_subset):
        x, cls, _, _ = orig[s["path"]]
        if method == "gradcam++":
            g = GradCAMPlusPlus(m, m.attn); c, _ = g.generate(x, class_idx=cls); g.remove()
        else:
            sc = ScoreCAM(m, m.attn); c, _ = sc.generate(x, class_idx=cls); sc.remove()
        out[s["path"]] = c
    return out

def agree(cams, method):
    idx = 2 if method == "gradcam++" else 3
    r = [compute_method_agreement(orig[p][idx], c) for p, c in cams.items()]
    return float(np.mean([a["spearman_rho"] for a in r])), float(np.mean([a["iou_top_k"] for a in r]))

rows, final_cams = [], {}
for method in ["gradcam++", "scorecam"]:
    for seed in SEEDS_RAND:
        # cascade
        gen = torch.Generator(device="cpu").manual_seed(1000 + seed); mc = copy.deepcopy(model).eval()
        for stage, blk in enumerate(CASCADE_BLOCKS, 1):
            _randomize_block_(getattr(mc, blk), gen)
            rho, iou = agree(cams_for(mc, method), method)
            rows.append({"method": method, "seed": seed, "mode": "cascade", "stage": stage, "block": blk, "rho": rho, "iou": iou})
        final_cams[(method, seed)] = cams_for(mc, method); del mc
        # each block alone
        for blk in CASCADE_BLOCKS:
            gen = torch.Generator(device="cpu").manual_seed(2000 + seed); m1 = copy.deepcopy(model).eval()
            _randomize_block_(getattr(m1, blk), gen)
            rho, iou = agree(cams_for(m1, method), method)
            rows.append({"method": method, "seed": seed, "mode": "single_block", "stage": 0, "block": blk, "rho": rho, "iou": iou}); del m1
        if DEVICE == "cuda": torch.cuda.empty_cache()
    print(method, "done")
casc = pd.DataFrame(rows); casc.to_csv(OUT_DIR / "randomization_cascade.csv", index=False)

summ_rows = []
for method in ["gradcam++", "scorecam"]:
    # chance level: agreement between maps of two different fully randomized models
    ch = []
    for a in SEEDS_RAND:
        for b in SEEDS_RAND:
            if a < b:
                ca, cb = final_cams[(method, a)], final_cams[(method, b)]
                ch.append(np.mean([compute_method_agreement(ca[p], cb[p])["spearman_rho"] for p in ca]))
    chance = float(np.mean(ch)); sub = casc[(casc.method == method) & (casc["mode"] == "cascade")]
    final = sub[sub.stage == len(CASCADE_BLOCKS)]
    rho_final, iou_final = final.rho.mean(), final.iou.mean()
    summ_rows.append({"method": method, "rho_after_full_cascade_mean": rho_final, "rho_sd_over_seeds": final.rho.std(ddof=1),
                      "iou_after_full_cascade_mean": iou_final, "chance_rho_two_random_models": chance,
                      "pass": bool(rho_final < 0.2 and abs(rho_final - chance) < 0.1)})
rs = pd.DataFrame(summ_rows); rs.to_csv(OUT_DIR / "randomization_summary.csv", index=False)
print(casc[casc["mode"] == "cascade"].groupby(["method", "stage", "block"])[["rho", "iou"]].mean().round(3).to_string())
print("\nsingle block randomized alone (mean over seeds):")
print(casc[casc["mode"] == "single_block"].groupby(["method", "block"])[["rho", "iou"]].mean().round(3).to_string())
print("\n", rs.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 4))
for method in ["gradcam++", "scorecam"]:
    g = casc[(casc.method == method) & (casc["mode"] == "cascade")].groupby("stage").rho.agg(["mean", "std"])
    ax.errorbar([0] + list(g.index), [1.0] + list(g["mean"]), yerr=[0] + list(g["std"].fillna(0)), marker="o", label=method)
ax.set_xticks(range(len(CASCADE_BLOCKS) + 1)); ax.set_xticklabels(["trained"] + CASCADE_BLOCKS, rotation=30)
ax.set_ylabel("Spearman rho vs trained-model map"); ax.legend(); ax.grid(alpha=.3); plt.tight_layout()
plt.savefig(OUT_DIR / "randomization_cascade_curve.png", dpi=150); plt.show()


## §7 · Attribution vs. ground-truth tumor masks (point 68)

BRISC2025 provides a segmentation mask for each image. For every tumor-class test image with a mask (Grad-CAM++ on all of them; Score-CAM and
LIME on the 32-image subset), three overlap measures are computed against the mask resized to 224 × 224 (nearest neighbour):
**pointing game** (the maximum of the map lies inside the mask), **top-15 % IoU** (the 15 % most attributed pixels vs. the mask) and
**energy in mask** (share of total attribution mass inside the mask). Two baselines are scored with the same code: a **random** map and a
**centre-Gaussian** map. The chance level of all three measures depends on mask size, so the mean mask area is reported alongside.

In [ ]:
seg_root = BRISC_ROOT.parent / "segmentation_task"
mask_index = {p.stem: p for p in seg_root.rglob("*.png") if "mask" in str(p.parent).lower()}
print("segmentation root:", seg_root, "| masks indexed:", len(mask_index))
assert len(mask_index) > 0, "no masks found: attach the BRISC2025 segmentation_task folder or fix seg_root"

def load_mask224(path):
    stem = Path(path).stem
    if stem not in mask_index: return None
    m = cv2.imread(str(mask_index[stem]), cv2.IMREAD_GRAYSCALE)
    if m is None: return None
    m = cv2.resize(m, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST) > 0
    return m if m.any() else None

def overlap(cam, mask, top=0.15):
    cam = np.asarray(cam, np.float64); k = max(1, int(round(top * cam.size)))
    thr = np.partition(cam.ravel(), -k)[-k]; top_mask = cam >= thr
    iou = (top_mask & mask).sum() / max((top_mask | mask).sum(), 1)
    point = bool(mask.flat[int(np.argmax(cam))])
    energy = float(cam[mask].sum() / max(cam.sum(), 1e-12))
    return point, float(iou), energy

yy, xx = np.mgrid[:IMG_SIZE, :IMG_SIZE]
centre_map = np.exp(-(((yy - IMG_SIZE/2)**2 + (xx - IMG_SIZE/2)**2) / (2 * (IMG_SIZE/4)**2)))
rng_m = np.random.default_rng(SEED)

tumor = [s for s in test_s if s["class_name"] != "notumor"]
rows_g = []
for s in tumor:
    mk = load_mask224(s["path"])
    if mk is None: continue
    x = load_input_tensor(s["path"])
    g = GradCAMPlusPlus(model, model.attn); cam, _ = g.generate(x); g.remove()
    for method, c in [("gradcam++", cam), ("random", rng_m.random((IMG_SIZE, IMG_SIZE))), ("centre_gaussian", centre_map)]:
        pt, iou, en = overlap(c, mk)
        rows_g.append({"class": s["class_name"], "method": method, "pointing": pt, "iou15": iou, "energy": en, "mask_frac": mk.mean()})
gdf = pd.DataFrame(rows_g)
print(f"tumor test images with a mask: {gdf[gdf.method=='gradcam++'].shape[0]} of {len(tumor)}")
res_all = gdf.groupby("method")[["pointing", "iou15", "energy", "mask_frac"]].mean().round(4)
res_cls = gdf.groupby(["class", "method"])[["pointing", "iou15", "energy", "mask_frac"]].mean().round(4)
print(res_all.to_string()); print(); print(res_cls.to_string())
res_all.to_csv(OUT_DIR / "mask_overlap_gradcampp_all_tumor_images.csv"); res_cls.to_csv(OUT_DIR / "mask_overlap_by_class.csv")

rows_s = []
for s in explain_subset:
    if s["class_name"] == "notumor": continue
    mk = load_mask224(s["path"])
    if mk is None: continue
    x, cls, cam_g, _ = None, None, None, None
    x = load_input_tensor(s["path"])
    g = GradCAMPlusPlus(model, model.attn); cam_g, cls = g.generate(x); g.remove()
    sc = ScoreCAM(model, model.attn); cam_s, _ = sc.generate(x, class_idx=cls); sc.remove()
    lime_exp = LimeImageExplainer(model).explain(x, class_idx=cls); cam_l = np.abs(lime_segments_to_dense_map(lime_exp))
    for method, c in [("gradcam++", cam_g), ("scorecam", cam_s), ("lime", cam_l)]:
        pt, iou, en = overlap(c, mk)
        rows_s.append({"class": s["class_name"], "method": method, "pointing": pt, "iou15": iou, "energy": en, "mask_frac": mk.mean()})
sdf = pd.DataFrame(rows_s)
res_sub = sdf.groupby("method")[["pointing", "iou15", "energy", "mask_frac"]].agg(["mean"]).round(4)
res_sub["n_images"] = sdf[sdf.method == "gradcam++"].shape[0]
print("\n32-image subset (tumor images only):"); print(res_sub.to_string())
sdf.to_csv(OUT_DIR / "mask_overlap_subset_per_image.csv", index=False)


## §8 · Done

Send back the files listed at the top (CSV and NPZ in `/kaggle/working`), or just the printed output of §3–§7.